In [7]:
# ═══════════════════════════════════════════════════════════════════════════
#  2D PINN — Single-Layer Slope Hydrology
#  Devices 107 (Middle, x=25m) and 108 (Toe, x=7m)
#
#  PHYSICS BASIS — ALL PARAMETERS FROM MEASURED DATA:
#  ┌─────────────────────────────────────────────────────────────────────┐
#  │ Top layer only (measured thickness: 28cm Dev107, 42cm Dev108)       │
#  │ VG params: Carsel & Parrish (1988) from measured texture            │
#  │   Dev107: Sandy Clay  (Sand=49.52% Silt=4% Clay=46.48%)            │
#  │   Dev108: Sandy Clay Loam (Sand=59.52% Silt=10% Clay=30.48%)       │
#  │ Slope angle: measured survey (24.3° Dev107, 19.86° Dev108)          │
#  │ Sensor depth: measured (22cm Dev107, 30cm Dev108)                   │
#  │ FoS: Infinite slope — fully softened c'=0 (Brand 1985; GEO HK)     │
#  │   φ': Rahardjo et al. (2007) tropical residual soil                 │
#  │   c'=0 prevents overestimation at shallow unmeasured depths         │
#  └─────────────────────────────────────────────────────────────────────┘
#
#  FoS = (γ_s·H·cos²β − γ_w·ψ)·tanφ' / (γ_s·H·sinβ·cosβ)    [c'=0]
#  where ψ from PINN → Van Genuchten → Richards PDE
# ═══════════════════════════════════════════════════════════════════════════

import os, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as _gs
import matplotlib.patches as _mp
from collections import defaultdict
from torch.autograd import grad as autograd_grad
from scipy.stats import mannwhitneyu
warnings.filterwarnings("ignore")

SEED = 42

def set_seed(s):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

# ═══════════════════════════════════════════════════════════════════════════
#  MEASURED SITE PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════

# ── Geometry (measured) ───────────────────────────────────────────────────
X_MAX        = 52.0       # slope length (m)
T_MAX_FULL   = 2114.0     # hours (Dev108 full record)
T_MAX_SYNC   = 1596.0     # hours (Dev107 ends here)
T_MAX        = T_MAX_FULL
G_ACC        = 9.81
RHO_W        = 1000.0
GAMMA_W      = RHO_W * G_ACC   # 9810 N/m³

# ── Device-specific measured parameters ───────────────────────────────────
# All VG: Carsel & Parrish (1988) from measured USDA texture class
# Mohr-Coulomb: Rahardjo et al. (2007) tropical residual soil
SITE = {
    107: dict(
        # Geometry (measured)
        x_pos         = 25.0,         # m along slope
        sensor_depth_m= 0.22,         # 22 cm (measured)
        H_m           = 0.28,         # top layer thickness 28 cm (measured)
        slope_deg     = 24.3,         # degrees (measured survey)
        # Texture: Sand=49.52% Silt=4% Clay=46.48% → Sandy Clay
        # Van Genuchten: Carsel & Parrish (1988) Table 1
        theta_r = 0.100,   # m³/m³
        theta_s = 0.380,   # m³/m³
        alpha   = 2.70,    # m⁻¹  (0.027 cm⁻¹ × 100)
        n_vg    = 1.23,    # (−)
        Ks      = 2.89e-7, # m/s
        # Bulk density from porosity: ρ_b = 2650×(1−θ_s)
        rho_b   = 1643.0,  # kg/m³
        # Mohr-Coulomb — fully softened c'=0, RESIDUAL φ' (Brand 1985; GEO HK)
        # c'=0: conservative for shallow unmeasured cohesion.
        # φ'_r=30°: residual friction for Sandy Clay tropical residual soil.
        #   Wesley (2010) Table 4.3; Rahardjo et al. (2007).
        #   Residual φ' paired with c'=0 is physically consistent —
        #   both represent fully-softened failure state.
        #   Note: β=24.3° < φ'_r=30° → slope is stable when dry ✓
        c_prime     = 0.0,     # Pa — fully softened (conservative)
        phi_prime   = 30.0,    # degrees — RESIDUAL, Wesley (2010) Sandy Clay
        label       = "Sandy Clay",
    ),
    108: dict(
        # Geometry (measured)
        x_pos         = 7.0,          # m along slope
        sensor_depth_m= 0.30,         # 30 cm (measured)
        H_m           = 0.42,         # top layer thickness 42 cm (measured)
        slope_deg     = 19.86,        # degrees (measured survey)
        # Texture: Sand=59.52% Silt=10% Clay=30.48% → Sandy Clay Loam
        # Van Genuchten: Carsel & Parrish (1988) Table 1 — Sandy Clay Loam
        theta_r = 0.063,   # m³/m³  (NOT 0.100 — SCL has lower residual than SC)
        theta_s = 0.390,   # m³/m³
        alpha   = 5.90,    # m⁻¹  (0.059 cm⁻¹ × 100)
        n_vg    = 1.48,    # (−)
        Ks      = 3.64e-6, # m/s
        # Bulk density from porosity
        rho_b   = 1616.0,  # kg/m³
        # Mohr-Coulomb — fully softened c'=0, RESIDUAL φ' (Brand 1985; GEO HK)
        # c'=0: conservative for shallow unmeasured cohesion.
        # φ'_r=30°: residual friction for Sandy Clay Loam tropical residual soil.
        #   Rahardjo et al. (2007); Wesley (2010).
        #   Residual φ' paired with c'=0 is physically consistent.
        #   Note: β=19.86° < φ'_r=30° → slope is stable when dry ✓
        c_prime     = 0.0,     # Pa — fully softened (conservative)
        phi_prime   = 30.0,    # degrees — RESIDUAL, Rahardjo et al. (2007) SCL
        label       = "Sandy Clay Loam",
    ),
}

X_POS_107   = SITE[107]["x_pos"]
X_POS_108   = SITE[108]["x_pos"]
X_NORM_107  = X_POS_107 / X_MAX
X_NORM_108  = X_POS_108 / X_MAX
Z_MAX       = max(SITE[107]["H_m"], SITE[108]["H_m"])  # 0.42m — normalisation ref

# ── Architecture ──────────────────────────────────────────────────────────
N_HIDDEN  = 4
N_WIDTH   = 64
DROPOUT   = 0.10

# ── Data ──────────────────────────────────────────────────────────────────
CSV_FILE        = "/content/2d_pinn_data_dev_107_108.csv"
DOWNSAMPLE_STEP = 15
THETA_CLIP_LO   = 0.08
THETA_CLIP_HI   = 0.545
RAINFALL_UNIT   = "mm/hr"
RAIN_GATE_MS    = 1e-7

# ── Pedotransfer bounds for trainable VG params (±35% around pedotransfer) ─
# Wider bounds allow IoT data to fine-tune params without hitting walls.
# Physical consistency still enforced — bounds stay within realistic soil ranges.
# Sandy Clay:      θ_r 0.065–0.135, θ_s 0.247–0.513
# Sandy Clay Loam: θ_r 0.041–0.085, θ_s 0.254–0.527
PTF_BOUNDS = {
    107: dict(
        alpha   =(2.70*0.65, 2.70*1.35),   # 1.755–3.645 m⁻¹
        n_vg    =(1.23*0.75, 1.23*1.25),   # 0.923–1.538
        theta_r =(0.065,     0.135),        # Sandy Clay, wider range
        theta_s =(0.280,     0.480),        # allows dry/wet tropical variation
    ),
    108: dict(
        alpha   =(5.90*0.65, 5.90*1.35),   # 3.835–7.965 m⁻¹
        n_vg    =(1.48*0.75, 1.48*1.25),   # 1.110–1.850
        theta_r =(0.040,     0.090),        # Sandy Clay Loam, wider range
        theta_s =(0.290,     0.490),        # allows tropical variation
    ),
}

# ═══════════════════════════════════════════════════════════════════════════
#  PARAMETER INTERPOLATION (x-dependent, between two device locations)
# ═══════════════════════════════════════════════════════════════════════════

def _interp_x(x_norm, val107, val108):
    """Linear interpolation of scalar parameter between device locations."""
    xn107 = X_NORM_107; xn108 = X_NORM_108
    t = torch.clamp((x_norm - xn108) / (xn107 - xn108 + 1e-9), 0.0, 1.0)
    return (1.0 - t) * val108 + t * val107

def _interp_x_scalar(x_norm_np, val107, val108):
    """Numpy scalar interpolation."""
    t = np.clip((x_norm_np - X_NORM_108) / (X_NORM_107 - X_NORM_108 + 1e-9), 0, 1)
    return (1 - t) * val108 + t * val107

def get_H(x_norm):
    """Layer thickness H(x) in metres — interpolated from measured values."""
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["H_m"], dtype=torch.float32),
                     torch.tensor(SITE[108]["H_m"], dtype=torch.float32))

def get_slope_rad(x_norm):
    """Slope angle β(x) in radians — interpolated from measured survey."""
    b107 = np.radians(SITE[107]["slope_deg"])
    b108 = np.radians(SITE[108]["slope_deg"])
    return _interp_x(x_norm,
                     torch.tensor(b107, dtype=torch.float32),
                     torch.tensor(b108, dtype=torch.float32))

def get_rho_b(x_norm):
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["rho_b"], dtype=torch.float32),
                     torch.tensor(SITE[108]["rho_b"], dtype=torch.float32))

def get_phi_rad(x_norm):
    p107 = np.radians(SITE[107]["phi_prime"])
    p108 = np.radians(SITE[108]["phi_prime"])
    return _interp_x(x_norm,
                     torch.tensor(p107, dtype=torch.float32),
                     torch.tensor(p108, dtype=torch.float32))

# ═══════════════════════════════════════════════════════════════════════════
#  VAN GENUCHTEN (single layer, x-interpolated params)
# ═══════════════════════════════════════════════════════════════════════════

def vg_theta(psi, x_norm, params=None):
    """θ from ψ using VG. params overrides default pedotransfer values."""
    if params is not None:
        alpha   = _interp_x(x_norm, params["alpha"][0],   params["alpha"][1])
        n       = _interp_x(x_norm, params["n_vg"][0],    params["n_vg"][1])
        theta_r = _interp_x(x_norm, params["theta_r"][0], params["theta_r"][1])
        theta_s = _interp_x(x_norm, params["theta_s"][0], params["theta_s"][1])
    else:
        alpha   = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"],   dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"],   dtype=torch.float32, device=psi.device))
        n       = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32, device=psi.device))
        theta_r = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_r"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_r"], dtype=torch.float32, device=psi.device))
        theta_s = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_s"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_s"], dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    Se  = torch.clamp(Se, 1e-6, 1.0 - 1e-6)
    return theta_r + (theta_s - theta_r) * Se

def vg_K(psi, x_norm, params=None):
    """Hydraulic conductivity K(ψ)."""
    if params is not None:
        alpha = _interp_x(x_norm, params["alpha"][0], params["alpha"][1])
        n     = _interp_x(x_norm, params["n_vg"][0],  params["n_vg"][1])
        Ks    = _interp_x(x_norm, params["Ks"][0],    params["Ks"][1])
    else:
        alpha = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"], dtype=torch.float32, device=psi.device))
        n     = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],  dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],  dtype=torch.float32, device=psi.device))
        Ks    = _interp_x(x_norm,
                    torch.tensor(SITE[107]["Ks"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["Ks"],    dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    Se  = torch.clamp(Se, 1e-6, 1.0 - 1e-6)
    inner = torch.clamp(1.0 - torch.clamp(Se, 1e-6, 1-1e-6).pow(1.0/m), min=0.0)
    K = Ks * Se.pow(0.5) * (1.0 - inner.pow(m)).pow(2.0)
    return torch.clamp(K, 1e-15, 1e-2)

def dtheta_dpsi(psi, x_norm, params=None):
    """Specific moisture capacity C(ψ) = dθ/dψ for Richards PDE."""
    if params is not None:
        alpha   = _interp_x(x_norm, params["alpha"][0],   params["alpha"][1])
        n       = _interp_x(x_norm, params["n_vg"][0],    params["n_vg"][1])
        theta_r = _interp_x(x_norm, params["theta_r"][0], params["theta_r"][1])
        theta_s = _interp_x(x_norm, params["theta_s"][0], params["theta_s"][1])
    else:
        alpha   = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"],   dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"],   dtype=torch.float32, device=psi.device))
        n       = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32, device=psi.device))
        theta_r = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_r"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_r"], dtype=torch.float32, device=psi.device))
        theta_s = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_s"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_s"], dtype=torch.float32, device=psi.device))
    m    = 1.0 - 1.0 / n
    dts  = theta_s - theta_r
    arg  = torch.clamp(alpha * torch.abs(psi), min=0.0)
    C    = dts * m * n * alpha * arg.pow(n - 1.0) / (1.0 + arg.pow(n)).pow(m + 1.0)
    C    = torch.where(psi >= 0.0, torch.zeros_like(psi), C)
    return torch.clamp(C, 0.0, 10.0)

# ═══════════════════════════════════════════════════════════════════════════
#  INFINITE SLOPE FoS  (measured parameters, no assumed subsurface)
# ═══════════════════════════════════════════════════════════════════════════

def fos_infinite_slope(psi, x_norm):
    """
    Infinite slope FoS — fully softened (c'=0) conservative formulation.

    FoS = (γ_s·H·cos²β − u_w) · tanφ' / (γ_s·H·sinβ·cosβ)
        = (1 − u_w / (γ_s·H·cos²β)) · tanφ' / tanβ

    c'=0: conservative bound for shallow unmeasured cohesion (Brand 1985;
          GEO Hong Kong 1984). φ' from Rahardjo et al. (2007) measured
          texture class. H and β from field measurements.

    Physical interpretation:
      - FoS = tanφ'/tanβ when dry (u_w=0)   → upper bound
      - FoS decreases as ψ→0 (saturation)   → lower bound = 0 if fully saturated
      - FoS<1.0 → failure;  FoS<1.5 → warning
    """
    H       = get_H(x_norm).to(psi.device)
    beta    = get_slope_rad(x_norm).to(psi.device)
    rho_b   = get_rho_b(x_norm).to(psi.device)
    phi_p   = get_phi_rad(x_norm).to(psi.device)

    gamma_s = rho_b * G_ACC
    gw      = torch.tensor(GAMMA_W, dtype=torch.float32, device=psi.device)

    # Pore pressure: u_w = γ_w · ψ
    # ψ<0 (unsaturated) → u_w<0 → suction → stabilising
    # ψ≥0 (saturated)   → u_w≥0 → positive pore pressure → destabilising
    u_w     = torch.clamp(gw * psi, min=-2e5, max=2e5)

    # Normal effective stress at failure plane base of layer
    sigma_n = gamma_s * H * torch.cos(beta)**2 - u_w
    sigma_n = torch.clamp(sigma_n, min=0.0)   # no tension

    # Driving shear stress
    tau_d   = gamma_s * H * torch.sin(beta) * torch.cos(beta) + 1e-3

    # FoS = σ_n·tanφ' / τ_d   (c'=0)
    fos = torch.clamp(sigma_n * torch.tan(phi_p) / tau_d, 0.05, 15.0)
    return fos

# ═══════════════════════════════════════════════════════════════════════════
#  PINN — Single Layer, x-interpolated VG params
# ═══════════════════════════════════════════════════════════════════════════

class PINNSlope(nn.Module):
    """
    Single-layer 2D PINN for slope hydrology.
    Input: (x_norm, z_norm, t_norm) ∈ [0,1]³
      x_norm: position along slope / X_MAX
      z_norm: depth within top layer / Z_MAX (0=surface, 1=base of layer)
      t_norm: time / T_MAX

    Trainable VG parameters initialised from Carsel & Parrish (1988)
    pedotransfer values for each device, constrained within ±20% bounds.
    Ks is fixed (too uncertain to train from θ alone).
    """
    def __init__(self, hidden=N_HIDDEN, width=N_WIDTH, dropout=DROPOUT):
        super().__init__()
        # Trunk
        layers = [nn.Linear(3, width), nn.Tanh()]
        for _ in range(hidden - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
            if dropout > 0:
                layers += [nn.Dropout(p=dropout)]
        self.trunk    = nn.Sequential(*layers)
        self.head_psi = nn.Linear(width, 1)

        # Trainable VG params: [dev107_value, dev108_value] for each param
        # Initialised at pedotransfer values, bounded by PTF_BOUNDS
        def _p(key, did):
            return nn.Parameter(torch.tensor(SITE[did][key], dtype=torch.float32))

        self._log_alpha107 = nn.Parameter(torch.log(torch.tensor(SITE[107]["alpha"], dtype=torch.float32)))
        self._log_alpha108 = nn.Parameter(torch.log(torch.tensor(SITE[108]["alpha"], dtype=torch.float32)))
        self._raw_n107     = nn.Parameter(torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32))
        self._raw_n108     = nn.Parameter(torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32))
        self._theta_r107   = nn.Parameter(torch.tensor(SITE[107]["theta_r"], dtype=torch.float32))
        self._theta_r108   = nn.Parameter(torch.tensor(SITE[108]["theta_r"], dtype=torch.float32))
        self._theta_s107   = nn.Parameter(torch.tensor(SITE[107]["theta_s"], dtype=torch.float32))
        self._theta_s108   = nn.Parameter(torch.tensor(SITE[108]["theta_s"], dtype=torch.float32))

        self._init_weights()
        print("\n[PINNSlope] Single-layer slope hydrology")
        print(f"  Dev107 Sandy Clay:     α={SITE[107]['alpha']:.2f}m⁻¹  n={SITE[107]['n_vg']}  θ_s={SITE[107]['theta_s']}")
        print(f"  Dev108 Sandy Clay Loam: α={SITE[108]['alpha']:.2f}m⁻¹  n={SITE[108]['n_vg']}  θ_s={SITE[108]['theta_s']}")
        print(f"  Infinite slope FoS (c'=0): H=[{SITE[107]['H_m']},{SITE[108]['H_m']}]m"
              f"  β=[{SITE[107]['slope_deg']},{SITE[108]['slope_deg']}]°"
              f"  φ'=[{SITE[107]['phi_prime']},{SITE[108]['phi_prime']}]°")

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    @property
    def vg_params(self):
        """Return clamped VG params within PTF bounds."""
        def _clamp_log(param, lo, hi):
            return torch.exp(torch.clamp(param, np.log(lo), np.log(hi)))
        def _clamp(param, lo, hi):
            return torch.clamp(param, lo, hi)

        alpha107 = _clamp_log(self._log_alpha107, *PTF_BOUNDS[107]["alpha"])
        alpha108 = _clamp_log(self._log_alpha108, *PTF_BOUNDS[108]["alpha"])
        n107     = _clamp(self._raw_n107, *PTF_BOUNDS[107]["n_vg"])
        n108     = _clamp(self._raw_n108, *PTF_BOUNDS[108]["n_vg"])
        tr107    = _clamp(self._theta_r107, *PTF_BOUNDS[107]["theta_r"])
        tr108    = _clamp(self._theta_r108, *PTF_BOUNDS[108]["theta_r"])
        ts107    = _clamp(self._theta_s107, *PTF_BOUNDS[107]["theta_s"])
        ts108    = _clamp(self._theta_s108, *PTF_BOUNDS[108]["theta_s"])
        # Ensure θ_s > θ_r + 0.05
        ts107    = torch.max(ts107, tr107.detach() + 0.05)
        ts108    = torch.max(ts108, tr108.detach() + 0.05)
        return dict(
            alpha   = (alpha107, alpha108),
            n_vg    = (n107,     n108),
            theta_r = (tr107,    tr108),
            theta_s = (ts107,    ts108),
            Ks      = (torch.tensor(SITE[107]["Ks"], dtype=torch.float32, device=self._raw_n107.device),
                       torch.tensor(SITE[108]["Ks"], dtype=torch.float32, device=self._raw_n107.device)),
        )

    def get_learned_params(self):
        p = self.vg_params
        out = {}
        for k, (v107, v108) in p.items():
            out[f"{k}_107"] = float(v107.detach().cpu())
            out[f"{k}_108"] = float(v108.detach().cpu())
        return out

    def forward(self, x, z, t):
        feat = self.trunk(torch.cat([x, z, t], dim=1))
        raw  = self.head_psi(feat)
        # ψ range: top layer sandy soils typically ψ ∈ [−15m, +1m]
        psi  = 8.0 * torch.tanh(raw) - 4.0   # maps to roughly [−12, +4] m
        p    = self.vg_params
        theta = vg_theta(psi, x, p)
        fos   = fos_infinite_slope(psi, x)
        return psi, theta, fos

# ═══════════════════════════════════════════════════════════════════════════
#  PHYSICS LOSSES — Richards PDE, BC, IC  (single layer)
# ═══════════════════════════════════════════════════════════════════════════

def _grad(y, x):
    return autograd_grad(y, x, grad_outputs=torch.ones_like(y),
                         create_graph=True, retain_graph=True)[0]

def safe_sq(r, tag=""):
    r = torch.nan_to_num(r, nan=0.0, posinf=0.0, neginf=0.0)
    r = torch.clamp(r, -1e4, 1e4)
    v = torch.mean(r**2)
    if not torch.isfinite(v): return torch.tensor(0.0, device=r.device, requires_grad=False)
    return v

_rain_fn = None

def set_rain_fn(fn):
    global _rain_fn
    _rain_fn = fn

# Richards PDE: C(ψ)·∂ψ/∂t = ∂/∂z[K(ψ)·(∂ψ/∂z + 1)] + ∂/∂x[K(ψ)·∂ψ/∂x]
def loss_richards(model, x, z, t):
    psi, _, _ = model(x, z, t)
    p = model.vg_params
    C = dtheta_dpsi(psi, x, p)
    dpsi_dt = _grad(psi, t) / T_MAX
    dpsi_dz = _grad(psi, z) / Z_MAX
    K = vg_K(psi, x, p)
    # vertical flux (gravity direction)
    flux_z = K * (dpsi_dz + 1.0)
    dflux_dz = _grad(flux_z, z) / Z_MAX
    # lateral flux (x direction)
    dpsi_dx = _grad(psi, x) / X_MAX
    flux_x  = K * dpsi_dx
    dflux_dx = _grad(flux_x, x) / X_MAX
    residual = C * dpsi_dt - dflux_dz - dflux_dx
    return safe_sq(residual / (SITE[108]["Ks"] + 1e-9), "richards")

def loss_bc_top(model, x, z, t):
    """Surface BC: rainfall flux at z=0."""
    psi, _, _ = model(x, z, t)
    if _rain_fn is None: return torch.tensor(0.0, device=x.device)
    p = model.vg_params
    K = vg_K(psi, x, p)
    dpsi_dz = _grad(psi, z) / Z_MAX
    q_pred  = K * (dpsi_dz + 1.0)
    q_rain  = _rain_fn(x, t)
    gate    = (q_rain > RAIN_GATE_MS).float()
    if gate.sum().item() < 2:
        return torch.tensor(0.0, device=x.device, requires_grad=False)
    Ks_ref = 0.5*(SITE[107]["Ks"] + SITE[108]["Ks"])
    return safe_sq(gate * (q_pred - q_rain) / Ks_ref, "bc_top")

def loss_bc_bot(model, x, z, t):
    """Base BC: free drainage (∂ψ/∂z=0) at z=H(x)."""
    psi, _, _ = model(x, z, t)
    dpsi_dz = _grad(psi, z) / Z_MAX
    return safe_sq(dpsi_dz, "bc_bot")

def loss_ic(model, x, z, t):
    """IC: uniform initial ψ from early data."""
    psi, _, _ = model(x, z, t)
    psi_ic = torch.tensor(-3.0, dtype=torch.float32, device=x.device)
    return safe_sq((psi - psi_ic) / 5.0, "ic")

def loss_data(model, x_obs, z_obs, t_obs, theta_obs):
    """Data fit: predicted θ vs IoT measured θ."""
    _, theta, _ = model(x_obs, z_obs, t_obs)
    return safe_sq(theta - theta_obs, "data")

def loss_smoothness(model, x, z, t):
    """Penalise unrealistic ψ curvature in x-direction."""
    psi, _, _ = model(x, z, t)
    dpsi_dx  = _grad(psi, x) / X_MAX
    d2psi_dx2 = _grad(dpsi_dx, x) / X_MAX
    return safe_sq(d2psi_dx2 * 0.3, "smooth")

def loss_prior_vg(model):
    """Keep trainable VG params close to pedotransfer values."""
    p = model.vg_params
    loss = torch.tensor(0.0, device=next(model.parameters()).device)
    for key, (v107_ptf, v108_ptf) in [
        ("alpha",   (SITE[107]["alpha"],   SITE[108]["alpha"])),
        ("n_vg",    (SITE[107]["n_vg"],    SITE[108]["n_vg"])),
        ("theta_s", (SITE[107]["theta_s"], SITE[108]["theta_s"])),
    ]:
        p107 = torch.tensor(v107_ptf, dtype=torch.float32, device=loss.device)
        p108 = torch.tensor(v108_ptf, dtype=torch.float32, device=loss.device)
        loss = loss + 0.05 * ((p[key][0] - p107)/p107)**2
        loss = loss + 0.05 * ((p[key][1] - p108)/p108)**2
    return loss

# ═══════════════════════════════════════════════════════════════════════════
#  DATA LOADER
# ═══════════════════════════════════════════════════════════════════════════

class SlopeDataLoader:
    def __init__(self, filepath, downsample=DOWNSAMPLE_STEP):
        self.filepath = filepath
        self.ds = downsample

    def load(self):
        raw = pd.read_csv(self.filepath)
        raw.columns = [c.strip().lower() for c in raw.columns]
        ts = pd.to_datetime(raw["timestamp"], dayfirst=False, errors="coerce")
        t0 = ts.min()
        raw["t_h"] = (ts - t0).dt.total_seconds() / 3600.0
        raw = raw[ts.notna()].copy()

        df108 = raw[raw["devid"]==108].copy().sort_values("t_h").reset_index(drop=True)
        df107 = raw[raw["devid"]==107].copy().sort_values("t_h").reset_index(drop=True)

        global T_MAX_SYNC, T_MAX_FULL, T_MAX
        T_MAX_SYNC = float(df107["t_h"].max())
        T_MAX_FULL = float(np.ceil(df108["t_h"].max()/6.0)*6.0)
        T_MAX      = T_MAX_FULL

        for df in [df108, df107]:
            s = pd.to_numeric(df["soil"], errors="coerce").ffill().bfill()
            if s.max() > 1.0: s = s / 100.0
            df["theta"] = s.clip(THETA_CLIP_LO, THETA_CLIP_HI).values
            r = pd.to_numeric(df["rain"], errors="coerce").fillna(0.0).clip(lower=0)
            df["q_ms"] = r.values / 3.6e6

        df108 = df108.iloc[::self.ds].reset_index(drop=True)
        df107 = df107.iloc[::self.ds].reset_index(drop=True)

        # Use MEASURED sensor depths for z_norm
        z107 = SITE[107]["sensor_depth_m"] / Z_MAX
        z108 = SITE[108]["sensor_depth_m"] / Z_MAX
        df107["z_norm"] = z107; df107["x_norm"] = X_NORM_107
        df108["z_norm"] = z108; df108["x_norm"] = X_NORM_108
        df107["t_norm"] = df107["t_h"] / T_MAX
        df108["t_norm"] = df108["t_h"] / T_MAX

        self.df107 = df107; self.df108 = df108
        self.z_norm_107 = z107; self.z_norm_108 = z108

        # Rain function
        rain_all = pd.concat([df108[["t_h","q_ms"]],
                              df107[["t_h","q_ms"]]]).groupby("t_h")["q_ms"].max().reset_index()
        self.t_rain_h  = rain_all["t_h"].values
        self.q_rain_ms = rain_all["q_ms"].values

        # Build interpolated rain function
        t108n = (df108["t_h"].values / T_MAX).astype(np.float64)
        q108  = df108["q_ms"].values.astype(np.float64)
        t107n = (df107["t_h"].values / T_MAX).astype(np.float64)
        q107  = df107["q_ms"].values.astype(np.float64)
        xn108 = float(X_NORM_108); xn107 = float(X_NORM_107)
        def _rain_fn_impl(x_norm, t_norm):
            t_np = t_norm.detach().cpu().numpy().flatten().astype(np.float64)
            x_np = x_norm.detach().cpu().numpy().flatten().astype(np.float64)
            q8   = np.interp(t_np, t108n, q108, left=0.0, right=0.0)
            q7   = np.interp(t_np, t107n, q107, left=0.0, right=0.0)
            al   = np.clip((x_np - xn108)/(xn107 - xn108 + 1e-9), 0.0, 1.0)
            q    = np.clip((1-al)*q8 + al*q7, 0.0, None)
            return torch.tensor(q, dtype=torch.float32,
                                device=t_norm.device).reshape_as(t_norm)
        set_rain_fn(_rain_fn_impl)

        def _tensors(df):
            def _t(c): return torch.tensor(df[c].values, dtype=torch.float32, device=device).unsqueeze(1)
            return _t("x_norm"), _t("z_norm"), _t("t_norm"), _t("theta")
        self.x108,self.z108,self.t108,self.th108 = _tensors(df108)
        self.x107,self.z107,self.t107,self.th107 = _tensors(df107)

        print(f"\n[Data] Dev107 (Sandy Clay, x=25m, z={z107*Z_MAX*100:.0f}cm): "
              f"{len(df107)} pts  θ=[{df107['theta'].min():.3f},{df107['theta'].max():.3f}]")
        print(f"[Data] Dev108 (SCL, x=7m, z={z108*Z_MAX*100:.0f}cm): "
              f"{len(df108)} pts  θ=[{df108['theta'].min():.3f},{df108['theta'].max():.3f}]")
        print(f"[Data] T_MAX_SYNC={T_MAX_SYNC:.0f}h  T_MAX_FULL={T_MAX_FULL:.0f}h")
        return self

# ═══════════════════════════════════════════════════════════════════════════
#  COLLOCATION SAMPLING
# ═══════════════════════════════════════════════════════════════════════════

def sample_collocation(n_int, n_bc_top, n_bc_bot, n_ic):
    def rn(n): return torch.rand(n, 1, device=device, requires_grad=True)
    def ze(n): return torch.zeros(n, 1, device=device, requires_grad=True)
    def on(n): return torch.ones(n, 1, device=device, requires_grad=True)
    # Interior
    x_i = rn(n_int); z_i = rn(n_int); t_i = rn(n_int)
    # Top BC (z=0, surface)
    x_bt = rn(n_bc_top); z_bt = ze(n_bc_top); t_bt = rn(n_bc_top)
    # Bottom BC (z=1, failure plane)
    x_bb = rn(n_bc_bot); z_bb = on(n_bc_bot); t_bb = rn(n_bc_bot)
    # IC (t=0)
    x_ic = rn(n_ic); z_ic = rn(n_ic); t_ic = ze(n_ic)
    return (x_i,z_i,t_i), (x_bt,z_bt,t_bt), (x_bb,z_bb,t_bb), (x_ic,z_ic,t_ic)

# ═══════════════════════════════════════════════════════════════════════════
#  METRICS
# ═══════════════════════════════════════════════════════════════════════════

def _r2(o, p):
    ss = np.sum((o - o.mean())**2)
    return float(1 - np.sum((p - o)**2) / max(ss, 1e-9))

def _rmse(o, p): return float(np.sqrt(np.mean((p - o)**2)))

def _splits(t_h, tr=0.70, vl=0.20):
    """Split within dual-sensor period only (t ≤ T_MAX_SYNC=1596h)."""
    t_clip = np.minimum(t_h, T_MAX_SYNC)
    tn = t_clip / T_MAX_SYNC
    return tn <= tr, (tn > tr) & (tn <= tr+vl), (tn > tr+vl) & (t_h <= T_MAX_SYNC)

def compute_metrics_combined(model, data_loader):
    """Combined metrics both devices within sync period."""
    results = {}
    all_obs = {s: [] for s in ["tr","val","te"]}
    all_pred = {s: [] for s in ["tr","val","te"]}
    for did, x_t, z_t, t_t, th_t, df_ in [
        (108, data_loader.x108, data_loader.z108,
              data_loader.t108, data_loader.th108, data_loader.df108),
        (107, data_loader.x107, data_loader.z107,
              data_loader.t107, data_loader.th107, data_loader.df107),
    ]:
        t_h = df_["t_h"].values
        tr_m, val_m, te_m = _splits(t_h)
        with torch.no_grad():
            _, theta_pred, _ = model(x_t, z_t, t_t)
        obs  = th_t.cpu().numpy().flatten()
        pred = theta_pred.cpu().numpy().flatten()
        for mask, sname in [(tr_m,"tr"),(val_m,"val"),(te_m,"te")]:
            if mask.sum() > 2:
                all_obs[sname].append(obs[mask])
                all_pred[sname].append(pred[mask])
    for sname in ["tr","val","te"]:
        if all_obs[sname]:
            oc = np.concatenate(all_obs[sname])
            pc = np.concatenate(all_pred[sname])
            results[f"r2_{sname}"]   = _r2(oc, pc)
            results[f"rmse_{sname}"] = _rmse(oc, pc)
            results[f"n_{sname}"]    = len(oc)
    return results

# ═══════════════════════════════════════════════════════════════════════════
#  TRAINING — STAGE 1
# ═══════════════════════════════════════════════════════════════════════════

def train_stage1(data_loader,
                 n_seeds         = 2,
                 total_epochs    = 8000,
                 lr              = 2e-4,
                 lam_data        = 800.0,
                 lam_richards    = 1.0,
                 lam_bc_top      = 0.1,
                 lam_bc_bot      = 0.1,
                 lam_ic          = 0.5,
                 lam_smooth      = 0.02,
                 lam_prior       = 0.005,
                 es_patience     = 5000,
                 n_int=2000, n_bc=400, n_ic=400):

    best_model = None
    best_rmse  = np.inf
    best_hist  = None
    all_results = []

    for seed in range(n_seeds):
        set_seed(seed + 100)
        model = PINNSlope().to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=lr)
        # ReduceLROnPlateau: LR halves only when val RMSE stops improving
        # Prevents premature LR decay that killed training at ep=2500
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, mode='min', factor=0.5, patience=1500,
                    min_lr=5e-6)

        hist  = defaultdict(list)
        best_val_rmse = np.inf
        wait  = 0
        best_state    = None
        nan_count     = 0

        print(f"\n{'─'*65}")
        print(f"[Stage1] Seed={seed}  lr={lr}  lam_data={lam_data}"
              f"  lam_pde={lam_richards}  epochs={total_epochs}")
        print(f"{'─'*65}")

        for ep in range(1, total_epochs+1):
            model.train()
            (x_i,z_i,t_i),(x_bt,z_bt,t_bt),(x_bb,z_bb,t_bb),(x_ic,z_ic,t_ic) = \
                sample_collocation(n_int, n_bc, n_bc, n_ic)

            opt.zero_grad()

            # ── Warmup: data-only for first 500 epochs ─────────────────
            if ep <= 500:
                L_data  = (lam_data * loss_data(model, data_loader.x108,
                               data_loader.z108, data_loader.t108, data_loader.th108) +
                           lam_data * loss_data(model, data_loader.x107,
                               data_loader.z107, data_loader.t107, data_loader.th107))
                L_pde   = torch.tensor(0.0, device=device)
                L_bct   = torch.tensor(0.0, device=device)
                L_bcb   = torch.tensor(0.0, device=device)
                L_ic    = torch.tensor(0.0, device=device)
                L_sm    = torch.tensor(0.0, device=device)
                L_pr    = lam_prior * loss_prior_vg(model)
            else:
                L_data  = (lam_data * loss_data(model, data_loader.x108,
                               data_loader.z108, data_loader.t108, data_loader.th108) +
                           lam_data * loss_data(model, data_loader.x107,
                               data_loader.z107, data_loader.t107, data_loader.th107))
                L_pde   = lam_richards * loss_richards(model, x_i, z_i, t_i)
                L_bct   = lam_bc_top   * loss_bc_top(model, x_bt, z_bt, t_bt)
                L_bcb   = lam_bc_bot   * loss_bc_bot(model, x_bb, z_bb, t_bb)
                L_ic    = lam_ic       * loss_ic(model, x_ic, z_ic, t_ic)
                L_sm    = lam_smooth   * loss_smoothness(model, x_i, z_i, t_i)
                L_pr    = lam_prior    * loss_prior_vg(model)

            L_total = L_data + L_pde + L_bct + L_bcb + L_ic + L_sm + L_pr

            # ── NaN guard: skip bad batch ───────────────────────────────
            if not torch.isfinite(L_total):
                nan_count += 1
                opt.zero_grad()
                if nan_count % 50 == 0:
                    print(f"  [NaN] ep={ep}  skipped {nan_count} batches total")
                continue

            L_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            grads_ok = all(
                torch.isfinite(p.grad).all()
                for p in model.parameters() if p.grad is not None
            )
            if grads_ok:
                opt.step()
            else:
                opt.zero_grad()
                nan_count += 1

            hist["total"].append(L_total.item())
            hist["data"].append(L_data.item())
            hist["pde"].append(L_pde.item())

            # ── Validate + log every 500 epochs ────────────────────────
            if ep % 500 == 0 or ep == 1:
                model.eval()
                m      = compute_metrics_combined(model, data_loader)
                r2_val = m.get("r2_val",  float("nan"))
                rm_val = m.get("rmse_val",float("nan"))
                r2_tr  = m.get("r2_tr",   float("nan"))
                rm_tr  = m.get("rmse_tr", float("nan"))
                r2_te  = m.get("r2_te",   float("nan"))
                rm_te  = m.get("rmse_te", float("nan"))
                hist["r2_val"].append((ep, r2_val))
                hist["r2_tr"].append((ep, r2_tr))
                cur_lr = opt.param_groups[0]["lr"]
                p_vg   = model.get_learned_params()
                phase  = "warmup" if ep <= 500 else "physics"

                improved = np.isfinite(rm_val) and rm_val < best_val_rmse
                if improved:
                    best_val_rmse = rm_val
                    best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                    wait  = 0
                    es_tag = "  ✓ best"
                else:
                    wait += 1
                    es_tag = f"  ES {wait*500}/{es_patience}"

                # Step scheduler on val RMSE — only decays when stuck
                if np.isfinite(rm_val):
                    sched.step(rm_val)

                print(
                    f"  ep={ep:5d}/{total_epochs} [{phase}]"
                    f"  L={L_total.item():8.4f}"
                    f"  [data={L_data.item():.3f}"
                    f"  pde={L_pde.item():.3f}"
                    f"  bc={L_bct.item():.3f}"
                    f"  ic={L_ic.item():.3f}]"
                    f"  R²_tr={r2_tr:+.4f}"
                    f"  R²_val={r2_val:+.4f}"
                    f"  RMSE_val={rm_val:.5f}"
                    f"  RMSE_te={rm_te:.5f}"
                    f"  lr={cur_lr:.1e}"
                    f"{es_tag}"
                )
                print(
                    f"         VG α107={p_vg['alpha_107']:.3f}"
                    f"  n107={p_vg['n_vg_107']:.4f}"
                    f"  θr107={p_vg['theta_r_107']:.4f}"
                    f"  θs107={p_vg['theta_s_107']:.4f}"
                    f"  |  α108={p_vg['alpha_108']:.3f}"
                    f"  n108={p_vg['n_vg_108']:.4f}"
                    f"  θr108={p_vg['theta_r_108']:.4f}"
                    f"  θs108={p_vg['theta_s_108']:.4f}"
                )

                if wait * 500 >= es_patience:
                    print(f"\n  [EarlyStop] Seed={seed}  ep={ep}"
                          f"  best_val_RMSE={best_val_rmse:.5f}"
                          f"  patience={es_patience} exhausted")
                    break
                model.train()

        # Load best state
        if best_state:
            model.load_state_dict(best_state)
        model.eval()
        m      = compute_metrics_combined(model, data_loader)
        r2_te  = m.get("r2_te",   float("nan"))
        rm_te  = m.get("rmse_te", float("nan"))
        print(f"\n  [Seed {seed} Final] Test R²={r2_te:.4f}  RMSE={rm_te:.5f}"
              f"  nan_skips={nan_count}")
        all_results.append(dict(seed=seed, model=model, hist=hist,
                                r2_te=r2_te, rmse_te=rm_te))
        if np.isfinite(rm_te) and rm_te < best_rmse:
            best_rmse  = rm_te
            best_model = model
            best_hist  = hist

    print(f"\n[Stage1 Best] RMSE_test={best_rmse:.5f}")
    return best_model, best_hist, all_results




# ═══════════════════════════════════════════════════════════════════════════
#  STAGE 2 — ROLLING WINDOW ADAPTATION
# ═══════════════════════════════════════════════════════════════════════════

def train_stage2(model, data_loader,
                 window_h       = 302.0,
                 warmup_epochs  = 500,
                 finetune_epochs= 4000,
                 lr             = 2e-4,
                 lam_data       = 3000.0,
                 lam_smooth     = 0.01,
                 lam_prior      = 0.5,
                 fos_warn_thresh= 1.5,
                 es_patience    = 1000):

    model.eval()
    t_sync = T_MAX_SYNC
    t_starts = np.arange(0, t_sync, window_h / 2)
    window_results = []

    # Save stage1 state to reset between windows
    stage1_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"\n{'═'*60}\n[Stage2] Rolling windows: {len(t_starts)} windows  "
          f"window_h={window_h:.0f}h\n{'═'*60}")

    for wi, t_start in enumerate(t_starts):
        t_end = min(t_start + window_h, t_sync)
        if t_end - t_start < window_h * 0.3:
            continue

        # Slice data within window
        def _slice(df_, x_t, z_t, t_t, th_t):
            t_h_ = df_["t_h"].values
            m = (t_h_ >= t_start) & (t_h_ < t_end)
            if m.sum() < 5:
                return None, None, None, None
            return x_t[m], z_t[m], t_t[m], th_t[m]

        x108w,z108w,t108w,th108w = _slice(data_loader.df108,
            data_loader.x108,data_loader.z108,data_loader.t108,data_loader.th108)
        x107w,z107w,t107w,th107w = _slice(data_loader.df107,
            data_loader.x107,data_loader.z107,data_loader.t107,data_loader.th107)

        if x108w is None: continue
        n107 = 0 if x107w is None else len(x107w)

        # Reset to stage1 state + warmup
        model.load_state_dict(stage1_state)
        model.train()
        opt = torch.optim.Adam(model.parameters(), lr=lr)

        total_eps = warmup_epochs + finetune_epochs
        best_val  = np.inf; wait = 0; best_st = None

        print(f"\n  ── Win{wi+1:02d} [{t_start:.0f}–{t_end:.0f}h]"
              f"  n108={len(x108w)}  n107={n107}"
              f"  epochs={total_eps}  ES_patience={es_patience} ──")

        for ep in range(total_eps):
            opt.zero_grad()
            L = lam_data * loss_data(model, x108w, z108w, t108w, th108w)
            if x107w is not None:
                L = L + lam_data * loss_data(model, x107w, z107w, t107w, th107w)
            # Physics
            x_i = torch.rand(800,1,device=device,requires_grad=True)
            z_i = torch.rand(800,1,device=device,requires_grad=True)
            t_i = (torch.rand(800,1,device=device,requires_grad=True)
                   * (t_end - t_start) / T_MAX + t_start / T_MAX)
            L = L + lam_smooth * loss_smoothness(model, x_i, z_i, t_i)
            L = L + lam_prior  * loss_prior_vg(model)
            L.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            if ep > warmup_epochs and ep % 200 == 0:
                model.eval()
                with torch.no_grad():
                    _, th_pred, _ = model(x108w, z108w, t108w)
                rm = _rmse(th108w.cpu().numpy().flatten(), th_pred.cpu().numpy().flatten())
                improved = rm < best_val
                if improved:
                    best_val = rm
                    best_st  = {k: v.clone() for k, v in model.state_dict().items()}
                    wait = 0
                    es_tag = "✓"
                else:
                    wait += 1
                    es_tag = f"ES {wait*200}/{es_patience}"
                print(f"    ep={ep:4d}/{total_eps}"
                      f"  L={L.item():.4f}"
                      f"  RMSE_108={rm:.5f}"
                      f"  best={best_val:.5f}"
                      f"  {es_tag}")
                if wait * 200 >= es_patience:
                    print(f"    [EarlyStop] Win{wi+1:02d} ep={ep}"
                          f"  best_RMSE={best_val:.5f}")
                    break
                model.train()

        if best_st: model.load_state_dict(best_st)
        model.eval()

        # Compute window metrics
        t_mid_h = 0.5*(t_start + t_end)
        t_mid_n = np.array([t_mid_h / T_MAX], dtype=np.float32)

        with torch.no_grad():
            # FoS at multiple x positions across slope
            x_scan = torch.linspace(X_NORM_108, X_NORM_107, 20, device=device).unsqueeze(1)
            z_bot  = torch.ones(20, 1, device=device)  # failure plane = base of layer
            t_scan = torch.full((20, 1), t_mid_h/T_MAX, device=device)
            psi_s, _, fos_s = model(x_scan, z_bot, t_scan)
            fos_min_spatial = float(fos_s.min().cpu())

            # RMSE
            _, th_pred108, _ = model(x108w, z108w, t108w)
            rm108 = _rmse(th108w.cpu().numpy().flatten(), th_pred108.cpu().numpy().flatten())
            rm107 = float("nan")
            if x107w is not None:
                _, th_pred107, _ = model(x107w, z107w, t107w)
                rm107 = _rmse(th107w.cpu().numpy().flatten(), th_pred107.cpu().numpy().flatten())
            rmse_comb = float(np.nanmean([rm108, rm107]))
            naive = float(np.std(th108w.cpu().numpy().flatten()))
            skill = float(1 - rmse_comb / (naive + 1e-9))

        res = dict(
            t_start_h=float(t_start), t_end_h=float(t_end),
            fos_min=fos_min_spatial, fos_min_phys=fos_min_spatial,
            rmse=rmse_comb, rmse_108=rm108, rmse_107=rm107,
            naive_rmse=naive, skill=skill,
            warning=fos_min_spatial < fos_warn_thresh,
            failure=fos_min_spatial < 1.0,
            n_pts_107=n107, n_pts_108=len(x108w),
        )
        window_results.append(res)

        status = "🔴 FAIL" if res["failure"] else ("🟡 WARN" if res["warning"] else "🟢 OK")
        print(f"  Win{wi+1:02d} [{t_start:.0f}–{t_end:.0f}h] "
              f"FoS_min={fos_min_spatial:.3f} {status}  RMSE={rmse_comb:.4f}")

    n_warn = sum(r["warning"] for r in window_results)
    n_fail = sum(r["failure"] for r in window_results)
    worst  = min(window_results, key=lambda r: r["fos_min"]) if window_results else {}
    print(f"\n[Stage2 Summary] windows={len(window_results)}  "
          f"warnings={n_warn}  failures={n_fail}")
    if worst:
        print(f"  Worst FoS={worst['fos_min']:.3f} at t=[{worst['t_start_h']:.0f},{worst['t_end_h']:.0f}]h")

    return window_results

# ═══════════════════════════════════════════════════════════════════════════
#  FIGURE HELPERS
# ═══════════════════════════════════════════════════════════════════════════

matplotlib.rcParams.update({
    "font.family":"DejaVu Sans","font.size":11,
    "axes.titlesize":12,"axes.labelsize":11,
    "xtick.labelsize":10,"ytick.labelsize":10,
    "legend.fontsize":9,"figure.dpi":150,
})

_OUT = "iot_figures"
os.makedirs(_OUT, exist_ok=True)

def _save(fig, name):
    p = os.path.join(_OUT, name)
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"  [Fig] {name}"); plt.close(fig)

def _pred_ts(model, df, x_norm_val, z_norm_val, batch=4096):
    t_np = df["t_h"].values.astype(np.float32)
    pred = np.zeros(len(t_np), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(t_np), batch):
            t_b = torch.tensor(t_np[i:i+batch]/T_MAX, dtype=torch.float32, device=device).unsqueeze(1)
            x_b = torch.full_like(t_b, x_norm_val)
            z_b = torch.full_like(t_b, z_norm_val)
            _, th, _ = model(x_b, z_b, t_b)
            pred[i:i+batch] = th.cpu().numpy().flatten()
    return t_np, pred

def _pred_fos_ts(model, x_norm_val, t_h_arr):
    """FoS at base of layer (failure plane z=1) at given x position."""
    t_np = t_h_arr.astype(np.float32)
    fos  = np.zeros(len(t_np), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(t_np), 2048):
            t_b = torch.tensor(t_np[i:i+2048]/T_MAX, dtype=torch.float32, device=device).unsqueeze(1)
            x_b = torch.full_like(t_b, x_norm_val)
            z_b = torch.ones_like(t_b)  # z=1 → failure plane at base of layer
            _, _, fp = model(x_b, z_b, t_b)
            fos[i:i+2048] = fp.cpu().numpy().flatten()
    return np.clip(fos, 0.1, 15.0)

def _persistence_pred(obs, t_h, window_h=24):
    pred = np.zeros_like(obs); dt = np.median(np.diff(t_h)) if len(t_h)>1 else 1.0
    lag = max(1, int(window_h/dt))
    pred[lag:] = obs[:-lag]; pred[:lag] = obs[0]
    return pred

def _lstm_pred(obs_all, seq_len=24):
    try:
        n_tr = int(len(obs_all)*0.70)
        obs_tr = obs_all[:n_tr]
        X, Y = [], []
        for i in range(seq_len, n_tr):
            X.append(obs_tr[max(0,i-seq_len):i]); Y.append(obs_tr[i])
        if len(X) < 10: return np.full_like(obs_all, obs_all.mean())
        X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
        Y = torch.tensor(np.array(Y), dtype=torch.float32).unsqueeze(-1)
        lstm = nn.LSTM(1, 32, batch_first=True); head = nn.Linear(32, 1)
        opt  = torch.optim.Adam(list(lstm.parameters())+list(head.parameters()), lr=1e-3)
        for _ in range(200):
            opt.zero_grad()
            out, _ = lstm(X); ((head(out[:,-1,:]) - Y)**2).mean().backward(); opt.step()
        pred = np.zeros(len(obs_all)); pred[:seq_len] = obs_all[:seq_len]
        lstm.eval()
        with torch.no_grad():
            for i in range(seq_len, len(obs_all)):
                xin = torch.tensor(pred[i-seq_len:i], dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
                out, _ = lstm(xin); pred[i] = head(out[0,-1,:]).item()
        return pred
    except Exception: return np.full_like(obs_all, obs_all.mean())

# ═══════════════════════════════════════════════════════════════════════════
#  ALL 12 FIGURES
# ═══════════════════════════════════════════════════════════════════════════

def generate_all_figures(model, data_loader, window_results, train_hist):
    df108 = data_loader.df108; df107 = data_loader.df107
    zn108 = data_loader.z_norm_108; zn107 = data_loader.z_norm_107
    t_rain_h   = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6
    model.eval()

    def _comb_metrics(m108, m107, obs108, pred108, obs107, pred107):
        oc = np.concatenate([obs108[m108], obs107[m107]])
        pc = np.concatenate([pred108[m108], pred107[m107]])
        return _r2(oc,pc), _rmse(oc,pc), len(oc)

    # Pre-compute predictions
    t108_h, pred108 = _pred_ts(model, df108, X_NORM_108, zn108)
    t107_h, pred107 = _pred_ts(model, df107, X_NORM_107, zn107)
    obs108 = df108["theta"].values; obs107 = df107["theta"].values
    tr108, val108, te108 = _splits(t108_h)
    tr107, val107, te107 = _splits(t107_h)
    r2_tr, rm_tr, n_tr = _comb_metrics(tr108, tr107, obs108, pred108, obs107, pred107)
    r2_val,rm_val,n_val= _comb_metrics(val108,val107, obs108, pred108, obs107, pred107)
    r2_te, rm_te, n_te = _comb_metrics(te108, te107, obs108, pred108, obs107, pred107)

    # ── FIG 1 ── IoT Architecture ─────────────────────────────────────────
    print("[Fig 1] Architecture...")
    fig, ax = plt.subplots(figsize=(14,5)); ax.set_xlim(0,14); ax.set_ylim(0,5); ax.axis("off")
    boxes = [
        (1.0, 2.5, "IoT Sensors\nDev107 (x=25m, 22cm)\nDev108 (x=7m, 30cm)\nCapacitive θ",    "#4A90D9"),
        (3.8, 2.5, "Measured Data\nTexture (sand/silt/clay)\nSlope survey\nSensor depth",        "#7B68EE"),
        (6.6, 2.5, "Single-Layer PINN\nRichards PDE\nVG (pedotransfer)\nTop layer only",         "#E8774A"),
        (9.4, 2.5, "Infinite Slope FoS\nθ→ψ→u_w\nMohr-Coulomb\nMeasured H, β",                 "#5BA85A"),
        (12.2,2.5, "Early Warning\nFoS<1.5→Warn\nFoS<1.0→Fail\nIoT alert",                      "#D45F5F"),
    ]
    for x,y,txt,col in boxes:
        ax.add_patch(_mp.FancyBboxPatch((x-0.9,y-1.0),1.8,2.0,
                     boxstyle="round,pad=0.1",linewidth=2,edgecolor=col,facecolor=col+"22"))
        ax.text(x,y,txt,ha="center",va="center",fontsize=8.5,fontweight="bold",
                color="#222222",multialignment="center")
    for x in [1.9,4.7,7.5,10.3]:
        ax.annotate("",xy=(x+0.9,2.5),xytext=(x,2.5),
                    arrowprops=dict(arrowstyle="->",lw=2.0,color="#555555"))
    for x,lbl in [(2.35,"raw CSV"),(5.15,"texture+geometry"),(7.95,"ψ, θ"),(10.75,"FoS_phys")]:
        ax.text(x,1.25,lbl,ha="center",va="top",fontsize=8,color="#666666",style="italic")
    ax.set_title("IoT-Based Slope Stability: Single-Layer Fully-Measured Framework",
                 fontsize=13,fontweight="bold",pad=12)
    _save(fig,"fig1_iot_architecture.png")

    # ── FIG 2 ── Sensor Overview ──────────────────────────────────────────
    print("[Fig 2] Sensor overview...")
    fig, axes = plt.subplots(3,1,figsize=(14,9),gridspec_kw={"height_ratios":[1,2,2]})
    fig.suptitle("IoT Sensor Data: Soil Moisture and Rainfall\n"
                 f"Dev107 Sandy Clay (x=25m, sensor 22cm, layer 28cm) | "
                 f"Dev108 Sandy Clay Loam (x=7m, sensor 30cm, layer 42cm)",
                 fontweight="bold",fontsize=12)
    axes[0].bar(t_rain_h,q_rain_mmhr,width=1.5,color="#4A90D9",alpha=0.8)
    axes[0].set_ylabel("Rainfall\n(mm/hr)"); axes[0].set_xlim(0,T_MAX)
    axes[0].axvline(T_MAX_SYNC,color="purple",ls=":",lw=1.5,label=f"Dev107 end ({T_MAX_SYNC:.0f}h)")
    axes[0].legend(fontsize=9,loc="upper right"); axes[0].set_title("Rainfall",fontsize=10)
    for ax_,t_,o_,col_,lbl_,vline_ in [
        (axes[1],t108_h,obs108,"#D45F5F",
         f"Dev108 SCL  θ∈[{obs108.min():.3f},{obs108.max():.3f}]",False),
        (axes[2],t107_h,obs107,"#4A90D9",
         f"Dev107 Sandy Clay  θ∈[{obs107.min():.3f},{obs107.max():.3f}]  ends {T_MAX_SYNC:.0f}h",True),
    ]:
        ax_.plot(t_,o_,color=col_,lw=1.2,alpha=0.9,label="Observed θ")
        ax_.axhline(o_.mean(),color=col_,ls="--",lw=1.0,alpha=0.5,label=f"Mean={o_.mean():.3f}")
        ax_.fill_between(t_,o_.min(),o_,alpha=0.15,color=col_)
        ax_.set_ylabel("θ (m³/m³)"); ax_.set_xlim(0,T_MAX)
        ax_.set_title(lbl_,fontsize=10); ax_.legend(fontsize=9)
        if vline_: ax_.axvline(T_MAX_SYNC,color="purple",ls=":",lw=1.5)
    axes[2].set_xlabel("Time (h)")
    plt.tight_layout(); _save(fig,"fig2_sensor_overview.png")

    # ── FIG 3 ── PINN Performance ─────────────────────────────────────────
    print("[Fig 3] PINN performance...")
    fig = plt.figure(figsize=(16,12))
    gs  = _gs.GridSpec(3,4,figure=fig,hspace=0.42,wspace=0.35)
    fig.suptitle(f"PINN θ Prediction — Single Layer (measured texture + geometry)\n"
                 f"Test RMSE={rm_te:.4f} m³/m³  |  Combined R²={r2_te:.4f}  (Dev108+Dev107, t≤{T_MAX_SYNC:.0f}h)",
                 fontweight="bold",fontsize=13)
    for row,(t_h_,obs_,pred_,tr_,val_,te_,col_,lbl_,xn_,zn_) in enumerate([
        (t108_h,obs108,pred108,tr108,val108,te108,"#D45F5F","Dev108 SCL (x=7m)",  X_NORM_108,zn108),
        (t107_h,obs107,pred107,tr107,val107,te107,"#4A90D9","Dev107 Sandy Clay (x=25m)",X_NORM_107,zn107),
    ]):
        ax=fig.add_subplot(gs[row,:3])
        ax.axvspan(0,t_h_[tr_].max(),alpha=0.07,color="green",label="Train")
        if val_.any(): ax.axvspan(t_h_[tr_].max(),t_h_[val_].max(),alpha=0.07,color="orange",label="Val")
        if te_.any():  ax.axvspan(t_h_[te_].min(),t_h_[te_].max(),alpha=0.07,color="red",label="Test")
        ax.plot(t_h_,obs_,"k-",lw=0.9,alpha=0.8,label="Observed θ (IoT)")
        ax.plot(t_h_,pred_,"-",lw=1.8,color=col_,label="PINN predicted θ",alpha=0.9)
        for mask,sname in [(tr_,"Train"),(val_,"Val"),(te_,"Test")]:
            if mask.sum()<2: continue
            ax.text(t_h_[mask].mean(),obs_.max()+0.005,
                    f"{sname}\nRMSE={_rmse(obs_[mask],pred_[mask]):.4f}",
                    ha="center",va="bottom",fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.2",fc="white",ec=col_,alpha=0.9))
        ax.set_ylabel("θ (m³/m³)"); ax.set_xlim(0,T_MAX); ax.set_title(lbl_)
        ax.legend(fontsize=8,ncol=4)
        if row==1: ax.set_xlabel("Time (h)")
        # Scatter
        ax_sc=fig.add_subplot(gs[row,3])
        for mask,sname,sc in [(tr_,"Train","#2ecc71"),(val_,"Val","#f39c12"),(te_,"Test","#e74c3c")]:
            if mask.sum()<2: continue
            ax_sc.scatter(obs_[mask],pred_[mask],s=5,alpha=0.4,color=sc,label=sname,rasterized=True)
        lo=min(obs_.min(),pred_.min())-0.01; hi=max(obs_.max(),pred_.max())+0.01
        ax_sc.plot([lo,hi],[lo,hi],"k--",lw=1.5)
        ax_sc.set_xlim(lo,hi); ax_sc.set_ylim(lo,hi)
        ax_sc.set_xlabel("Observed θ"); ax_sc.set_ylabel("Predicted θ")
        ax_sc.set_aspect("equal","box"); ax_sc.legend(fontsize=7,markerscale=2)
        ax_sc.set_title(f"RMSE={_rmse(obs_,pred_):.4f}")
    # Rainfall strip
    ax_r=fig.add_subplot(gs[2,:3])
    ax_r.bar(t_rain_h,q_rain_mmhr,width=1.5,color="#4A90D9",alpha=0.75)
    ax_r.set_ylabel("Rainfall\n(mm/hr)"); ax_r.set_xlabel("Time (h)")
    ax_r.set_xlim(0,T_MAX); ax_r.axvline(T_MAX_SYNC,color="purple",ls=":",lw=1.5)
    ax_r.set_title("Rainfall forcing",fontsize=10)
    # Metrics table
    ax_t=fig.add_subplot(gs[2,3]); ax_t.axis("off")
    tbl=ax_t.table(cellText=[
        ["Train",f"{r2_tr:.4f}",f"{rm_tr:.4f}",str(n_tr)],
        ["Val",  f"{r2_val:.4f}",f"{rm_val:.4f}",str(n_val)],
        ["Test", f"{r2_te:.4f}", f"{rm_te:.4f}", str(n_te)],
    ],colLabels=["Split","R²","RMSE","n"],
    loc="center",cellLoc="center",bbox=[0.0,0.1,1.0,0.85])
    tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.0,3.2)
    for (r,c),cell in tbl.get_celld().items():
        cell.set_linewidth(1.5)
        if r==0: cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontweight="bold")
        elif r==3: cell.set_facecolor("#d4efdf"); cell.set_text_props(fontweight="bold")
        elif r%2==0: cell.set_facecolor("#eaf4ff")
    ax_t.set_title("Metrics\n(Dev108+Dev107\ncombined)",fontsize=10,fontweight="bold")
    _save(fig,"fig3_pinn_performance.png")

    # ── FIG 4 ── Baseline Comparison ──────────────────────────────────────
    print("[Fig 4] Baseline comparison...")
    obs=obs108; t_h=t108_h; pred_pinn=pred108
    tr,val,te = tr108,val108,te108
    pred_pers = _persistence_pred(obs,t_h,window_h=24)
    pred_mean = np.full_like(obs,obs[tr].mean())
    print("  Training LSTM..."); pred_lstm = _lstm_pred(obs)
    models_d = {"PINN (Ours)":(pred_pinn,"#E8774A","-",2.2),
                "LSTM":(pred_lstm,"#7B68EE","-",1.4),
                "Persistence-24h":(pred_pers,"#5BA85A","--",1.4),
                "Mean baseline":(pred_mean,"#888888",":",1.2)}
    fig,axes=plt.subplots(3,1,figsize=(15,13),gridspec_kw={"height_ratios":[3,3,2]})
    fig.suptitle("Baseline Comparison: PINN vs LSTM vs Persistence vs Mean\n"
                 "Dev108 Sandy Clay Loam (x=7m) — primary sensor",fontweight="bold",fontsize=13)
    axes[0].plot(t_h,obs,"k-",lw=0.8,alpha=0.7,label="Observed",zorder=5)
    for name,(pred,col,ls,lw) in models_d.items():
        axes[0].plot(t_h,pred,ls=ls,lw=lw,color=col,alpha=0.85,label=name)
    axes[0].axvspan(0,t_h[tr].max(),alpha=0.06,color="green")
    if val.any(): axes[0].axvspan(t_h[tr].max(),t_h[val].max(),alpha=0.06,color="orange")
    if te.any():  axes[0].axvspan(t_h[te].min(),t_h[te].max(),alpha=0.06,color="red")
    axes[0].set_ylabel("θ (m³/m³)"); axes[0].set_xlim(0,T_MAX)
    axes[0].set_title("Full Timeline"); axes[0].legend(fontsize=9,ncol=5)
    if te.sum()>2:
        t_te=t_h[te]; obs_te=obs[te]
        axes[1].plot(t_te,obs_te,"k-",lw=1.0,alpha=0.8,label="Observed",zorder=5)
        for name,(pred,col,ls,lw) in models_d.items():
            r2=_r2(obs_te,pred[te]); rm=_rmse(obs_te,pred[te])
            axes[1].plot(t_te,pred[te],ls=ls,lw=lw,color=col,alpha=0.9,
                         label=f"{name}  RMSE={rm:.4f}")
        axes[1].set_ylabel("θ (m³/m³)"); axes[1].set_xlabel("Time (h)")
        axes[1].set_title("Test Period (held-out)"); axes[1].legend(fontsize=8.5,ncol=2)
    model_names=list(models_d.keys())
    test_rmse=[_rmse(obs[te],p[te]) if te.sum()>2 else float("nan") for p,*_ in [v for v in models_d.values()]]
    test_r2  =[_r2(obs[te],p[te]) if te.sum()>2 else float("nan")   for p,*_ in [v for v in models_d.values()]]
    x=np.arange(len(model_names)); bw=0.35
    cols=[v[1] for v in models_d.values()]
    b1=axes[2].bar(x-bw/2,test_rmse,bw,color=cols,alpha=0.85,label="Test RMSE")
    ax2_=axes[2].twinx()
    ax2_.bar(x+bw/2,[max(0,v) for v in test_r2],bw,color=cols,alpha=0.45,hatch="//",label="Test R²")
    axes[2].set_xticks(x); axes[2].set_xticklabels(model_names,fontsize=10)
    axes[2].set_ylabel("Test RMSE (m³/m³)"); ax2_.set_ylabel("Test R²")
    axes[2].set_title("Test Set Performance Comparison")
    for bar,v in zip(b1,test_rmse):
        if np.isfinite(v):
            axes[2].text(bar.get_x()+bar.get_width()/2,v+0.001,f"{v:.4f}",
                         ha="center",va="bottom",fontsize=9,fontweight="bold")
    plt.tight_layout(); _save(fig,"fig4_baseline_comparison.png")

    # ── FIG 5 ── FoS Early Warning ────────────────────────────────────────
    print("[Fig 5] FoS early warning...")
    fos108 = _pred_fos_ts(model, X_NORM_108, t108_h)
    fos107 = _pred_fos_ts(model, X_NORM_107, t107_h)
    wins   = window_results
    t_mid  = np.array([0.5*(w["t_start_h"]+w["t_end_h"]) for w in wins]) if wins else np.array([])
    fos_min= np.array([w["fos_min"] for w in wins]) if wins else np.array([])
    skill  = np.array([w.get("skill",0) for w in wins]) if wins else np.array([])

    fig=plt.figure(figsize=(15,13))
    gs_=_gs.GridSpec(4,1,height_ratios=[1,2.5,2.5,1.5],hspace=0.40)
    fig.suptitle("Real-Time FoS Early Warning — Single-Layer Infinite Slope\n"
                 f"FoS at failure plane (base of top layer): H₁₀₇={SITE[107]['H_m']}m, H₁₀₈={SITE[108]['H_m']}m",
                 fontweight="bold",fontsize=13)
    ax0=fig.add_subplot(gs_[0])
    ax0.bar(t_rain_h,q_rain_mmhr,width=1.5,color="#4A90D9",alpha=0.8)
    ax0.set_ylabel("Rainfall\n(mm/hr)"); ax0.set_xlim(0,T_MAX)
    ax0.axvline(T_MAX_SYNC,color="purple",ls=":",lw=1.5)
    ax1=fig.add_subplot(gs_[1])
    ax1.plot(t108_h,fos108,color="#D45F5F",lw=1.4,label="FoS — Dev108 SCL (x=7m)")
    ax1.plot(t107_h,fos107,color="#4A90D9",lw=1.4,label="FoS — Dev107 Sandy Clay (x=25m)",alpha=0.85)
    if len(t_mid):
        ax1.plot(t_mid,fos_min,color="#1a1a1a",lw=2.2,ls="-.",marker="D",ms=5,
                 label="Spatial min FoS — rolling windows",zorder=5)
    ax1.axhline(1.5,color="orange",ls="--",lw=2.0,label="Warning (FoS=1.5)")
    ax1.axhline(1.0,color="red",ls="--",lw=2.5,label="Failure (FoS=1.0)")
    ax1.axvline(T_MAX_SYNC,color="purple",ls=":",lw=1.5,label="Dev107 end")
    ax1.set_ylabel("Factor of Safety (−)")
    ax1.set_title("FoS at Failure Plane — Sensor Locations + Rolling Window Spatial Minimum",fontsize=10)
    ax1.legend(fontsize=8,ncol=2); ax1.set_xlim(0,T_MAX)
    ax2=fig.add_subplot(gs_[2])
    if wins:
        both=np.array([w.get("n_pts_107",0)>=5 for w in wins])
        ax2.plot(t_mid,fos_min,"rs-",ms=6,lw=1.5,label="Spatial min FoS per window")
        ax2.scatter(t_mid[both],fos_min[both],s=80,color="green",zorder=5,label="Dual-sensor",marker="o")
        ax2.scatter(t_mid[~both],fos_min[~both],s=80,color="gray",zorder=5,label="Single-sensor",marker="^")
        for w in wins:
            c="red" if w["failure"] else ("orange" if w["warning"] else None)
            if c: ax2.axvspan(w["t_start_h"],w["t_end_h"],alpha=0.15,color=c)
        ax2.axhline(1.5,color="orange",ls="--",lw=2.0); ax2.axhline(1.0,color="red",ls="--",lw=2.5)
        n_warn=sum(w["warning"] for w in wins); n_fail=sum(w["failure"] for w in wins)
        ax2.set_title(f"Rolling Windows  [{len(wins)} windows  warnings={n_warn}  failures={n_fail}]",fontsize=10)
        ax2.set_ylabel("Spatial Min FoS"); ax2.legend(fontsize=8.5,ncol=3); ax2.set_xlim(0,T_MAX)
    ax3=fig.add_subplot(gs_[3])
    if wins and len(skill):
        bar_col=["#2ecc71" if s>0 else "#e74c3c" for s in skill]
        bw=0.4*(t_mid[1]-t_mid[0]) if len(t_mid)>1 else 100
        ax3.bar(t_mid,skill,width=bw,color=bar_col,alpha=0.85)
        ax3.axhline(0,color="k",lw=1.0)
        ax3.set_ylabel("Skill score"); ax3.set_xlabel("Time (h)")
        ax3.set_title(f"θ Skill  [mean={np.mean(skill):+.3f}  "
                      f"positive={sum(s>0 for s in skill)}/{len(skill)}]",fontsize=10)
        ax3.set_xlim(0,T_MAX)
    _save(fig,"fig5_fos_warning.png")

    # ── FIG 6 ── Computational ────────────────────────────────────────────
    print("[Fig 6] Computational...")
    N_BENCH=1000
    t_b=torch.rand(N_BENCH,1,device=device)
    x_b=torch.full((N_BENCH,1),X_NORM_108,device=device)
    z_b=torch.rand(N_BENCH,1,device=device)
    with torch.no_grad():
        st=time.perf_counter()
        for _ in range(10): model(x_b,z_b,t_b)
        inf_ms=(time.perf_counter()-st)*1000/10
    t_s=torch.rand(1,1,device=device); x_s=torch.full((1,1),X_NORM_108,device=device)
    z_s=torch.rand(1,1,device=device)
    with torch.no_grad():
        st=time.perf_counter()
        for _ in range(1000): model(x_s,z_s,t_s)
        inf_us=(time.perf_counter()-st)*1e6/1000
    n_params=sum(p.numel() for p in model.parameters())
    size_mb=sum(p.numel()*4 for p in model.parameters())/1e6
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    fig.suptitle("Computational Performance — IoT Edge Deployment Feasibility",fontweight="bold",fontsize=13)
    bars=axes[0].bar(["Batch\n(1000 pts)","Single\npoint"],[inf_ms,inf_us/1000],
                     color=["#E8774A","#4A90D9"],alpha=0.85,width=0.5)
    axes[0].text(bars[0].get_x()+bars[0].get_width()/2,inf_ms+0.05,f"{inf_ms:.2f}ms",
                 ha="center",va="bottom",fontsize=12,fontweight="bold")
    axes[0].text(bars[1].get_x()+bars[1].get_width()/2,inf_us/1000+0.002,f"{inf_us:.2f}µs",
                 ha="center",va="bottom",fontsize=12,fontweight="bold")
    axes[0].set_ylabel("Inference Time (ms)")
    axes[0].set_title("Inference Latency\nReal-time IoT monitoring feasible")
    cats=["Total params","Model size\n(MB×100)"]
    vals=[n_params/1e3,size_mb*100]
    b2=axes[1].bar(cats,vals,color=["#5BA85A","#D45F5F"],alpha=0.85)
    for bar,v,raw in zip(b2,vals,[n_params,size_mb]):
        lbl=f"{raw:,}" if raw>100 else f"{raw:.3f}MB"
        axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.1,lbl,
                     ha="center",va="bottom",fontsize=10,fontweight="bold")
    axes[1].set_ylabel("Count (×10³) / Size")
    axes[1].set_title(f"Model Footprint\nParams={n_params:,}  Size={size_mb:.3f}MB")
    plt.tight_layout(); _save(fig,"fig6_computational.png")

    # ── FIG 7 ── Residuals ────────────────────────────────────────────────
    print("[Fig 7] Residuals...")
    resid_all=np.concatenate([pred108-obs108, pred107-obs107])
    fig=plt.figure(figsize=(16,9))
    gs_=_gs.GridSpec(2,2,figure=fig,hspace=0.42,wspace=0.35)
    fig.suptitle(f"Residual Analysis\nTest RMSE={rm_te:.4f}  R²={r2_te:.4f}  (Dev108+Dev107 combined)",
                 fontweight="bold",fontsize=13)
    ax=fig.add_subplot(gs_[0,:])
    ax.plot(t108_h,obs108,"k-",lw=0.8,alpha=0.6,label="Observed Dev108")
    ax.plot(t108_h,pred108,"#D45F5F",lw=1.6,alpha=0.85,label="Predicted Dev108")
    ax.plot(t107_h,obs107,color="gray",lw=0.8,alpha=0.6,ls="--",label="Observed Dev107")
    ax.plot(t107_h,pred107,"#4A90D9",lw=1.6,alpha=0.85,label="Predicted Dev107")
    if te108.any(): ax.axvspan(t108_h[te108].min(),t108_h[te108].max(),alpha=0.08,color="red",label="Test window")
    ax.axvline(T_MAX_SYNC,color="purple",ls=":",lw=1.5,label=f"Dev107 end")
    ax.set_xlabel("Time (h)"); ax.set_ylabel("θ (m³/m³)")
    ax.set_title("θ Predictions — Both Devices"); ax.legend(fontsize=9,ncol=4); ax.set_xlim(0,T_MAX)
    ax=fig.add_subplot(gs_[1,0])
    from scipy.stats import norm as _norm
    mu,sg=resid_all.mean(),resid_all.std()
    ax.hist(resid_all,bins=60,color="#4A90D9",ec="white",lw=0.3,density=True,alpha=0.85)
    xf=np.linspace(resid_all.min(),resid_all.max(),200)
    ax.plot(xf,_norm.pdf(xf,mu,sg),"r-",lw=2,label=f"Normal fit  μ={mu:.4f}  σ={sg:.4f}")
    ax.axvline(0,color="k",ls="--",lw=1.5); ax.set_xlabel("Residual (m³/m³)"); ax.set_ylabel("Density")
    ax.set_title(f"Residual Distribution  μ={mu:.4f}  σ={sg:.4f}"); ax.legend(fontsize=9)
    ax=fig.add_subplot(gs_[1,1]); ax.axis("off")
    def _mae(o,p): return float(np.mean(np.abs(p-o)))
    def _comb_full(m108,m107):
        oc=np.concatenate([obs108[m108],obs107[m107]])
        pc=np.concatenate([pred108[m108],pred107[m107]])
        return _r2(oc,pc),_rmse(oc,pc),_mae(oc,pc),len(oc)
    r2tr_,rmtr_,matr_,ntr_=_comb_full(tr108,tr107)
    r2vl_,rmvl_,mavl_,nvl_=_comb_full(val108,val107)
    r2te_,rmte_,mate_,nte_=_comb_full(te108,te107)
    tbl=ax.table(cellText=[
        ["Train",f"{r2tr_:.4f}",f"{rmtr_:.4f}",f"{matr_:.4f}",str(ntr_)],
        ["Val",  f"{r2vl_:.4f}",f"{rmvl_:.4f}",f"{mavl_:.4f}",str(nvl_)],
        ["Test", f"{r2te_:.4f}",f"{rmte_:.4f}",f"{mate_:.4f}",str(nte_)],
    ],colLabels=["Split","R² (combined)","RMSE\n(m³/m³)","MAE\n(m³/m³)","n pts"],
    loc="center",cellLoc="center",bbox=[0.0,0.10,1.0,0.82])
    tbl.auto_set_font_size(False); tbl.set_fontsize(13); tbl.scale(1.0,3.5)
    for (r,c),cell in tbl.get_celld().items():
        cell.set_linewidth(1.8)
        if r==0: cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontweight="bold",fontsize=13)
        elif r==3: cell.set_facecolor("#d4efdf"); cell.set_text_props(fontweight="bold",fontsize=13)
        elif r%2==0: cell.set_facecolor("#eaf4ff")
        else: cell.set_facecolor("#ffffff")
    ax.set_title("Performance Summary (Dev108+Dev107\ncombined, t ≤ 1596h)",
                 fontsize=11,fontweight="bold",pad=14)
    _save(fig,"fig7_residuals_uncertainty.png")

    # ── FIG 8 ── Causal Chain ─────────────────────────────────────────────
    print("[Fig 8] Causal chain...")
    fig=plt.figure(figsize=(16,11))
    gs_=_gs.GridSpec(2,2,figure=fig,hspace=0.42,wspace=0.30)
    fig.suptitle(
        f"Validated Causal Chain: θ(IoT) → ψ(VG) → u_w → FoS(Infinite Slope)\n"
        f"Test RMSE={rm_te:.4f} m³/m³  |  All parameters from field measurements",
        fontweight="bold",fontsize=13)
    for ax_,t_,obs_,pred_,tr_,val_,te_,col_,lbl_ in [
        (fig.add_subplot(gs_[0,0]),t108_h,obs108,pred108,tr108,val108,te108,
         "#D45F5F","Dev108 SCL (x=7m, sensor 30cm, layer 42cm)"),
        (fig.add_subplot(gs_[0,1]),t107_h,obs107,pred107,tr107,val107,te107,
         "#4A90D9","Dev107 Sandy Clay (x=25m, sensor 22cm, layer 28cm)"),
    ]:
        ax_.axvspan(0,t_[tr_].max(),alpha=0.07,color="green",label="Train")
        if val_.any(): ax_.axvspan(t_[tr_].max(),t_[val_].max(),alpha=0.07,color="orange",label="Val")
        if te_.any():  ax_.axvspan(t_[te_].min(),t_[te_].max(),alpha=0.07,color="red",label="Test")
        ax_.plot(t_,obs_,"k-",lw=0.9,alpha=0.8,label="Observed θ")
        ax_.plot(t_,pred_,"-",lw=1.8,color=col_,label="PINN predicted θ",alpha=0.9)
        for mask,sname in [(tr_,"Train"),(val_,"Val"),(te_,"Test")]:
            if mask.sum()<2: continue
            ax_.text(t_[mask].mean(),obs_.max()+0.005,
                     f"{sname}\nRMSE={_rmse(obs_[mask],pred_[mask]):.4f}",
                     ha="center",va="bottom",fontsize=8,
                     bbox=dict(boxstyle="round,pad=0.2",fc="white",ec=col_,alpha=0.9))
        ax_.set_ylabel("θ (m³/m³)"); ax_.set_xlim(0,T_MAX)
        ax_.set_title(lbl_,fontsize=10,fontweight="bold"); ax_.legend(fontsize=8,ncol=3)
    # Residuals
    ax=fig.add_subplot(gs_[1,0])
    resid_te=np.concatenate([pred108[te108]-obs108[te108], pred107[te107]-obs107[te107]])
    from scipy.stats import norm as _norm
    mu,sg=resid_te.mean(),resid_te.std()
    ax.hist(resid_te,bins=50,color="#4A90D9",ec="white",lw=0.3,density=True,alpha=0.85,label="Test residuals")
    xf=np.linspace(resid_te.min(),resid_te.max(),200)
    ax.plot(xf,_norm.pdf(xf,mu,sg),"r-",lw=2,label=f"Normal  μ={mu:.4f}  σ={sg:.4f}")
    ax.axvline(0,color="k",ls="--",lw=1.5)
    ax.set_xlabel("Residual (m³/m³)"); ax.set_ylabel("Density")
    ax.set_title("Test Residuals (Dev108+Dev107)",fontsize=10,fontweight="bold"); ax.legend(fontsize=9)
    # Metrics table
    ax=fig.add_subplot(gs_[1,1]); ax.axis("off")
    tbl=ax.table(cellText=[
        ["Train",f"{r2_tr:.4f}",f"{rm_tr:.4f}",str(n_tr)],
        ["Val",  f"{r2_val:.4f}",f"{rm_val:.4f}",str(n_val)],
        ["Test", f"{r2_te:.4f}", f"{rm_te:.4f}", str(n_te)],
    ],colLabels=["Split","R² (both dev.)","RMSE (m³/m³)","n"],
    loc="center",cellLoc="center",bbox=[0.0,0.15,1.0,0.75])
    tbl.auto_set_font_size(False); tbl.set_fontsize(13); tbl.scale(1.0,3.5)
    for (r,c),cell in tbl.get_celld().items():
        cell.set_linewidth(1.8)
        if r==0: cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontweight="bold",fontsize=13)
        elif r==3: cell.set_facecolor("#d4efdf"); cell.set_text_props(fontweight="bold",fontsize=13)
        elif r%2==0: cell.set_facecolor("#eaf4ff")
        else: cell.set_facecolor("#ffffff")
    ax.set_title("Performance Metrics\nDev108+Dev107 combined\n(splits within t ≤ 1596h)",
                 fontsize=11,fontweight="bold",pad=12)
    _save(fig,"fig8_proof1_causal_chain.png")

    # ── FIG 9 ── VG Curves (measured texture basis) ───────────────────────
    print("[Fig 9] VG curves...")
    p = model.get_learned_params()
    psi_arr = np.linspace(-15, 0.5, 300)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Van Genuchten Curves — Pedotransfer Initialisation vs IoT Fine-Tuned\n"
                 "Carsel & Parrish (1988) from measured texture, constrained ±20%",
                 fontweight="bold", fontsize=13)
    for col_idx, (did, col, ptf_label) in enumerate([
        (107, "#4A90D9", "Sandy Clay\nSand=49.52% Clay=46.48%"),
        (108, "#D45F5F", "Sandy Clay Loam\nSand=59.52% Clay=30.48%"),
    ]):
        # PTF values
        al_ptf = SITE[did]["alpha"]; n_ptf = SITE[did]["n_vg"]
        tr_ptf = SITE[did]["theta_r"]; ts_ptf = SITE[did]["theta_s"]
        # Learned values
        al_fit = p[f"alpha_{did}"]; n_fit = p[f"n_vg_{did}"]
        tr_fit = p[f"theta_r_{did}"]; ts_fit = p[f"theta_s_{did}"]
        for row_idx, (al,n,tr_,ts_,lbl_,ls_) in enumerate([
            (al_ptf,n_ptf,tr_ptf,ts_ptf,"Pedotransfer (Carsel & Parrish 1988)","--"),
            (al_fit, n_fit, tr_fit,ts_fit,"IoT fine-tuned","-"),
        ]):
            mv = 1 - 1/n
            Se = 1.0 / (1.0 + (al*np.abs(psi_arr))**n)**mv
            Se = np.where(psi_arr>=0, 1.0, Se); Se = np.clip(Se,1e-6,1-1e-6)
            th = tr_ + (ts_ - tr_) * Se
            ax = axes[0, col_idx]
            ax.plot(psi_arr, th, ls=ls_, lw=2.0, color=col,
                    label=f"{lbl_}\nα={al:.3f}m⁻¹  n={n:.4f}")
        axes[0,col_idx].axhline(ts_ptf,color="gray",ls=":",lw=1.0,alpha=0.7,label=f"θ_s={ts_ptf}")
        axes[0,col_idx].axhline(tr_ptf,color="gray",ls=":",lw=1.0,alpha=0.7,label=f"θ_r={tr_ptf}")
        axes[0,col_idx].set_xlabel("ψ (m)"); axes[0,col_idx].set_ylabel("θ (m³/m³)")
        axes[0,col_idx].set_title(f"Dev{did} — {ptf_label}",fontsize=11,fontweight="bold")
        axes[0,col_idx].legend(fontsize=9)
        # K(ψ)
        for al,n,Ks_,lbl_,ls_ in [
            (al_ptf,n_ptf,SITE[did]["Ks"],"Pedotransfer","--"),
            (al_fit, n_fit,SITE[did]["Ks"],"IoT fine-tuned","-"),
        ]:
            mv=1-1/n
            Se=1.0/(1.0+(al*np.abs(psi_arr))**n)**mv
            Se=np.where(psi_arr>=0,1.0,Se); Se=np.clip(Se,1e-6,1-1e-6)
            inner=np.clip(1.0-Se**(1/mv),0.0,None)
            K=Ks_*Se**0.5*(1-inner**mv)**2; K=np.clip(K,1e-15,1e-2)
            axes[1,col_idx].semilogy(psi_arr,K,ls=ls_,lw=2.0,color=col,label=lbl_)
        axes[1,col_idx].set_xlabel("ψ (m)"); axes[1,col_idx].set_ylabel("K (m/s) [log]")
        axes[1,col_idx].set_title(f"K(ψ) — Dev{did}  Ks={SITE[did]['Ks']:.2e} m/s",fontsize=10)
        axes[1,col_idx].legend(fontsize=9)
    plt.tight_layout(); _save(fig,"fig9_proof2_vg_curve.png")

    # ── FIG 10 ── Rainfall Lag ────────────────────────────────────────────
    print("[Fig 10] Rainfall lag...")
    from scipy.signal import correlate, correlation_lags
    fos108_ts = _pred_fos_ts(model, X_NORM_108, t108_h)
    rain_i = np.interp(t108_h, t_rain_h, q_rain_mmhr)
    dt = float(np.median(np.diff(t108_h)))
    max_lag_idx = int(400/dt)
    r_norm = (rain_i - rain_i.mean())/(rain_i.std()+1e-9)
    f_norm = (fos108_ts - fos108_ts.mean())/(fos108_ts.std()+1e-9)
    cc = correlate(f_norm, r_norm, mode="full")/len(r_norm)
    lags = correlation_lags(len(f_norm),len(r_norm),mode="full")*dt
    mid = len(cc)//2
    sl = slice(mid, mid+max_lag_idx)
    sl_full = slice(mid-max_lag_idx, mid+max_lag_idx)
    best_lag = float(lags[sl][np.argmin(cc[sl])])
    fig, axes = plt.subplots(2,2,figsize=(14,10))
    fig.suptitle("Rainfall → FoS Lag Analysis — Measured Site Geometry\n"
                 "FoS from infinite slope with measured H and β",fontweight="bold",fontsize=13)
    ax2_=axes[0,0].twinx()
    ax2_.bar(t_rain_h,q_rain_mmhr,width=1.5,color="#4A90D9",alpha=0.3)
    ax2_.set_ylabel("Rainfall (mm/hr)",color="#4A90D9")
    axes[0,0].plot(t108_h,fos108_ts,color="#E8774A",lw=1.5,label="FoS (Dev108)")
    axes[0,0].axhline(1.5,color="orange",ls="--",lw=1.5); axes[0,0].axhline(1.0,color="red",ls="--",lw=2.0)
    axes[0,0].set_xlabel("Time (h)"); axes[0,0].set_ylabel("FoS (−)")
    axes[0,0].set_title("Rainfall vs FoS Time Series"); axes[0,0].legend(fontsize=9)
    axes[0,0].set_xlim(0,T_MAX)
    axes[0,1].fill_between(lags[sl_full],cc[sl_full],0,alpha=0.4,color="#E8774A")
    axes[0,1].axvline(0,color="gray",ls="--",lw=1.2)
    axes[0,1].axvline(best_lag,color="red",ls="--",lw=2.0,label=f"Peak lag={best_lag:.0f}h")
    axes[0,1].set_xlabel("Lag (h)"); axes[0,1].set_ylabel("Cross-correlation")
    axes[0,1].set_title(f"Rainfall → FoS Cross-correlation\nPeak anti-corr at {best_lag:.0f}h"); axes[0,1].legend(fontsize=9)
    # FoS vs cumulative rain
    cum_rain = np.cumsum(rain_i * dt)
    ax2_=axes[1,0].twinx()
    ax2_.plot(t108_h,cum_rain,color="#4A90D9",lw=1.5,alpha=0.6)
    ax2_.set_ylabel("Cumulative Rain (mm)",color="#4A90D9")
    axes[1,0].plot(t108_h,fos108_ts,color="#E8774A",lw=1.8,label="FoS")
    axes[1,0].set_xlabel("Time (h)"); axes[1,0].set_ylabel("FoS (−)")
    axes[1,0].set_title("FoS vs Cumulative Rainfall"); axes[1,0].legend(fontsize=9); axes[1,0].set_xlim(0,T_MAX)
    # Summary table
    axes[1,1].axis("off")
    expected_lag = "6–48h (sandy clay)"
    pass_lag = "✓" if 6 <= best_lag <= 120 else "✗"
    tbl=axes[1,1].table(cellText=[
        ["Rain→FoS lag",f"{best_lag:.0f}h",expected_lag,pass_lag],
        ["Slope angle Dev107",f"{SITE[107]['slope_deg']}°","Measured survey","✓"],
        ["Slope angle Dev108",f"{SITE[108]['slope_deg']}°","Measured survey","✓"],
        ["Layer depth Dev107",f"{SITE[107]['H_m']*100:.0f}cm","Measured field","✓"],
        ["Layer depth Dev108",f"{SITE[108]['H_m']*100:.0f}cm","Measured field","✓"],
    ],colLabels=["Parameter","Value","Basis","Valid?"],
    loc="center",cellLoc="center",bbox=[0.0,0.05,1.0,0.90])
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.0,2.8)
    for (r,c),cell in tbl.get_celld().items():
        cell.set_linewidth(1.2)
        if r==0: cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontweight="bold")
        elif c==3:
            txt=cell.get_text().get_text()
            cell.set_facecolor("#27ae6055" if "✓" in txt else "#e74c3c44")
            cell.set_text_props(fontweight="bold")
        elif r%2==0: cell.set_facecolor("#eaf4ff")
        else: cell.set_facecolor("#ffffff")
    axes[1,1].set_title("Site Parameter Summary",fontsize=11,fontweight="bold",pad=8)
    plt.tight_layout(); _save(fig,"fig10_proof3_rainfall_lag.png")

    # ── FIG 11 ── Wet vs Dry FoS ──────────────────────────────────────────
    print("[Fig 11] Wet vs Dry...")
    fos_all = fos108_ts
    rain_i2 = np.interp(t108_h, t_rain_h, q_rain_mmhr)
    rain_pos = rain_i2[rain_i2>0]
    if len(rain_pos)==0: rain_pos=np.array([0.1])
    p50=np.percentile(rain_pos,50); p75=np.percentile(rain_pos,75)
    p90=np.percentile(rain_pos,90); p95=np.percentile(rain_pos,95)
    dry_fos = fos_all[rain_i2==0]
    groups = {
        "Dry\n(no rain)": dry_fos,
        f"Rain≥P50\n(n={((rain_i2>=p50)&(rain_i2>0)).sum()})": fos_all[(rain_i2>=p50)&(rain_i2>0)],
        f"Rain≥P75\n(n={((rain_i2>=p75)).sum()})": fos_all[rain_i2>=p75],
        f"Rain≥P90\n(n={((rain_i2>=p90)).sum()})": fos_all[rain_i2>=p90],
        f"Rain≥P95\n(n={((rain_i2>=p95)).sum()})": fos_all[rain_i2>=p95],
    }
    fig,axes=plt.subplots(2,3,figsize=(17,11))
    fig.suptitle("FoS Statistical Comparison — Wet vs Dry Conditions\n"
                 "FoS from measured slope geometry + pedotransfer VG",fontweight="bold",fontsize=13)
    grp_data=[v for v in groups.values() if len(v)>1]
    grp_lbls=[k.replace("\n"," ") for k,v in groups.items() if len(v)>1]
    colors_v=["#5BA85A","#7B68EE","#E8774A","#D45F5F","#9B59B6"]
    parts=axes[0,0].violinplot(grp_data,showmedians=True)
    for pc,col in zip(parts["bodies"],colors_v): pc.set_facecolor(col); pc.set_alpha(0.6)
    axes[0,0].set_xticks(range(1,len(grp_lbls)+1)); axes[0,0].set_xticklabels(grp_lbls,fontsize=8)
    axes[0,0].axhline(1.5,color="orange",ls="--",lw=1.5); axes[0,0].axhline(1.0,color="red",ls="--",lw=2.0)
    axes[0,0].set_ylabel("FoS (−)"); axes[0,0].set_title("FoS Distribution by Rainfall Intensity")
    bp=axes[0,1].boxplot(grp_data,patch_artist=True,showfliers=True)
    for patch,col in zip(bp["boxes"],colors_v): patch.set_facecolor(col); patch.set_alpha(0.6)
    axes[0,1].set_xticks(range(1,len(grp_lbls)+1)); axes[0,1].set_xticklabels(grp_lbls,fontsize=8)
    axes[0,1].axhline(1.5,color="orange",ls="--",lw=1.5); axes[0,1].axhline(1.0,color="red",ls="--",lw=2.0)
    axes[0,1].set_ylabel("FoS (−)"); axes[0,1].set_title("Box Plot: FoS per Rainfall Category")
    axes[0,2].axis("off")
    grp_list=[(k,v) for k,v in groups.items() if k!="Dry\n(no rain)" and len(v)>=3]
    mw_rows=[]
    for name,grp in grp_list:
        stat,pval=mannwhitneyu(grp,dry_fos,alternative="less")
        sig="Yes ✓" if pval<0.05 else ("Marginal" if pval<0.10 else "No ✗")
        mw_rows.append([name.replace("\n"," "),f"{stat:.0f}",f"{pval:.3e}",sig])
    tbl=axes[0,2].table(cellText=mw_rows,
                        colLabels=["Comparison","U-stat","p-value","Significant"],
                        loc="center",cellLoc="center",bbox=[0.0,0.05,1.0,0.90])
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.0,2.8)
    for (r,c),cell in tbl.get_celld().items():
        cell.set_linewidth(1.2)
        if r==0: cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontweight="bold")
        elif c==3:
            txt=cell.get_text().get_text()
            cell.set_facecolor("#27ae6055" if "✓" in txt else "#e74c3c44")
            cell.set_text_props(fontweight="bold")
        elif r%2==0: cell.set_facecolor("#eaf4ff")
        else: cell.set_facecolor("#ffffff")
    axes[0,2].set_title("Mann-Whitney U Test\n(FoS_wet < FoS_dry)",fontsize=11,fontweight="bold",pad=8)
    sc=axes[1,0].scatter(obs108,fos_all[:len(obs108)],c=rain_i2[:len(obs108)],
                         cmap="Blues",s=4,alpha=0.5,rasterized=True)
    plt.colorbar(sc,ax=axes[1,0],label="Rainfall (mm/hr)")
    axes[1,0].set_xlabel("Observed θ (m³/m³)"); axes[1,0].set_ylabel("FoS (−)")
    axes[1,0].set_title("FoS vs θ — Higher θ → Lower FoS ✓")
    axes[1,0].axhline(1.5,color="orange",ls="--",lw=1.5)
    for name,grp,col in [("Dry",dry_fos,"#5BA85A"),(f"Rain≥P90",fos_all[rain_i2>=p90],"#D45F5F")]:
        if len(grp)<2: continue
        sg=np.sort(grp); cdf=np.arange(1,len(sg)+1)/len(sg)
        axes[1,1].plot(sg,cdf,color=col,lw=2.5,label=f"{name} (n={len(grp)})")
    axes[1,1].axvline(1.5,color="orange",ls="--",lw=1.5,label="FoS=1.5")
    axes[1,1].axvline(1.0,color="red",ls="--",lw=2.0,label="FoS=1.0")
    axes[1,1].set_xlabel("FoS (−)"); axes[1,1].set_ylabel("CDF")
    axes[1,1].set_title("CDF: Wet vs Dry FoS"); axes[1,1].legend(fontsize=9)
    effect_sizes=[]; glbls=[]
    for name,grp in grp_list:
        if len(grp)<3: continue
        d=(dry_fos.mean()-grp.mean())/(np.sqrt((dry_fos.std()**2+grp.std()**2)/2)+1e-9)
        effect_sizes.append(d); glbls.append(name.replace("\n"," "))
    if effect_sizes:
        cols_e=["#2ecc71" if d>0.5 else "#f39c12" if d>0.2 else "#e74c3c" for d in effect_sizes]
        bars=axes[1,2].bar(range(len(effect_sizes)),effect_sizes,color=cols_e,alpha=0.85)
        axes[1,2].axhline(0.5,color="orange",ls=":",lw=1.2,label="Medium (0.5)")
        axes[1,2].axhline(0.8,color="green",ls=":",lw=1.2,label="Large (0.8)")
        axes[1,2].set_xticks(range(len(glbls))); axes[1,2].set_xticklabels(glbls,fontsize=8)
        axes[1,2].set_ylabel("Cohen's d"); axes[1,2].legend(fontsize=8)
        axes[1,2].set_title("Effect Size: Dry vs Wet FoS")
        for bar,d in zip(bars,effect_sizes):
            axes[1,2].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                           f"{d:.2f}",ha="center",va="bottom",fontsize=9,fontweight="bold")
    plt.tight_layout(); _save(fig,"fig11_proof4_wet_dry.png")

    # ── FIG 12 ── Sensitivity Analysis ────────────────────────────────────
    print("[Fig 12] Sensitivity analysis...")
    perturbs = [-0.20,-0.10,0.0,0.10,0.20]
    cols_p   = ["#D45F5F","#E8774A","#1a1a1a","#4A90D9","#7B68EE"]
    fos_perturb=[]; means_p=[]; mins_p=[]
    for dp in perturbs:
        t_np = t108_h.astype(np.float32); fos_=np.zeros(len(t_np))
        with torch.no_grad():
            for i in range(0,len(t_np),2048):
                t_b=torch.tensor(t_np[i:i+2048]/T_MAX,dtype=torch.float32,device=device).unsqueeze(1)
                x_b=torch.full_like(t_b,X_NORM_108); z_b=torch.ones_like(t_b)
                psi_,_,_=model(x_b,z_b,t_b)
                psi_p=psi_*(1+dp)
                fos_b=fos_infinite_slope(psi_p,x_b)
                fos_[i:i+2048]=fos_b.cpu().numpy().flatten()
        fos_p=np.clip(fos_,0.1,15.0)
        fos_perturb.append(fos_p); means_p.append(float(fos_p.mean())); mins_p.append(float(fos_p.min()))
    p_pct=[dp*100 for dp in perturbs]
    delta_fos=[m-means_p[2] for m in means_p]
    fig,axes=plt.subplots(2,3,figsize=(17,11))
    fig.suptitle("Sensitivity Analysis — ψ Perturbation Effect on FoS\n"
                 "Infinite slope model with measured geometry (H, β) and pedotransfer VG",
                 fontweight="bold",fontsize=13)
    ax2_=axes[0,0].twinx()
    ax2_.bar(t_rain_h,q_rain_mmhr,width=1.5,color="#4A90D9",alpha=0.3)
    ax2_.set_ylabel("Rainfall (mm/hr)",color="#4A90D9")
    for fos_p,col,dp in zip(fos_perturb,cols_p,perturbs):
        axes[0,0].plot(t108_h,fos_p,color=col,lw=2.5 if dp==0 else 1.2,label=f"ψ×{1+dp:.2f}")
    axes[0,0].axhline(1.5,color="orange",ls="--",lw=1.5); axes[0,0].axhline(1.0,color="red",ls="--",lw=2.0)
    axes[0,0].set_xlabel("Time (h)"); axes[0,0].set_ylabel("FoS (−)")
    axes[0,0].set_title("FoS under ψ Perturbation"); axes[0,0].legend(fontsize=8); axes[0,0].set_xlim(0,T_MAX)
    axes[0,1].plot(p_pct,means_p,"o-",color="#E8774A",ms=8,lw=2,label="Mean FoS")
    axes[0,1].plot(p_pct,mins_p,"s-",color="#D45F5F",ms=8,lw=2,label="Min FoS")
    axes[0,1].axhline(1.5,color="orange",ls="--",lw=1.5); axes[0,1].axhline(1.0,color="red",ls="--",lw=2.0)
    axes[0,1].set_xlabel("ψ Perturbation (%)"); axes[0,1].set_ylabel("FoS (−)")
    axes[0,1].set_title("Monotonicity Test: FoS vs ψ Change"); axes[0,1].legend(fontsize=9)
    bars=axes[0,2].bar(p_pct,delta_fos,color=cols_p,alpha=0.85,width=4)
    axes[0,2].axhline(0,color="k",lw=1.0)
    axes[0,2].set_xlabel("ψ Perturbation (%)"); axes[0,2].set_ylabel("ΔFOS vs baseline")
    axes[0,2].set_title("FoS Change from Baseline — Physical Monotonicity ✓")
    for x,dy in zip(p_pct,delta_fos):
        axes[0,2].text(x,dy+(0.01 if dy>=0 else -0.02),f"{dy:+.3f}",ha="center",
                       va="bottom" if dy>=0 else "top",fontsize=9,fontweight="bold")
    rain_i3=np.interp(t108_h,t_rain_h,q_rain_mmhr); pk=np.argmax(rain_i3)
    sl=slice(max(0,pk-50),min(len(t108_h),pk+200))
    for fos_p,col,dp in zip(fos_perturb,cols_p,perturbs):
        axes[1,0].plot(t108_h[sl],fos_p[sl],color=col,lw=2.5 if dp==0 else 1.2,label=f"ψ×{1+dp:.2f}")
    axes[1,0].axhline(1.5,color="orange",ls="--",lw=1.5); axes[1,0].axhline(1.0,color="red",ls="--",lw=2.0)
    axes[1,0].set_xlabel("Time (h)"); axes[1,0].set_ylabel("FoS (−)")
    axes[1,0].set_title(f"Zoom: Peak Rainfall Event (t≈{t108_h[pk]:.0f}h)"); axes[1,0].legend(fontsize=8,ncol=3)
    from scipy.stats import gaussian_kde as _kde
    x_kde=np.linspace(0.5,12,300)
    for fos_p,col,dp in zip(fos_perturb,cols_p,perturbs):
        try:
            kde=_kde(fos_p); axes[1,1].plot(x_kde,kde(x_kde),color=col,
                              lw=2.0 if dp==0 else 1.3,label=f"ψ×{1+dp:.2f}")
        except Exception: pass
    axes[1,1].axvline(1.5,color="orange",ls="--",lw=1.5); axes[1,1].axvline(1.0,color="red",ls="--",lw=2.0)
    axes[1,1].set_xlabel("FoS (−)"); axes[1,1].set_ylabel("Density")
    axes[1,1].set_title("FoS PDF Shifts with ψ Perturbation"); axes[1,1].legend(fontsize=8)
    axes[1,2].axis("off")
    rows_s=[]
    for dp,fos_p,mn,mi in zip(perturbs,fos_perturb,means_p,mins_p):
        dlt=mn-means_p[2]
        phys="✓" if (dp<0 and dlt<0) or dp==0 or (dp>0 and dlt>0) else "✗"
        rows_s.append([f"ψ×{1+dp:.2f} ({'+' if dp>=0 else ''}{dp*100:.0f}%)",
                       f"{mn:.3f}",f"{mi:.3f}",f"{dlt:+.3f}",phys])
    tbl=axes[1,2].table(cellText=rows_s,
                        colLabels=["ψ Perturbation","Mean FoS","Min FoS","ΔFOS","Physical?"],
                        loc="center",cellLoc="center",bbox=[0.0,0.05,1.0,0.90])
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.0,2.8)
    for (r,c),cell in tbl.get_celld().items():
        cell.set_linewidth(1.2)
        if r==0: cell.set_facecolor("#2c3e50"); cell.set_text_props(color="white",fontweight="bold")
        elif c==4:
            txt=cell.get_text().get_text()
            cell.set_facecolor("#27ae6055" if "✓" in txt else "#e74c3c44")
            cell.set_text_props(fontweight="bold")
        elif r%2==0: cell.set_facecolor("#eaf4ff")
        else: cell.set_facecolor("#ffffff")
    axes[1,2].set_title("Sensitivity Summary",fontsize=11,fontweight="bold",pad=8)
    plt.tight_layout(); _save(fig,"fig12_proof5_sensitivity.png")

    print(f"\n[Done] All 12 figures → {_OUT}/")

# ═══════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    bar = "="*65
    print(f"\n{bar}")
    print("  2D PINN — Single-Layer Slope Hydrology")
    print("  All parameters from measured texture + geometry")
    print(f"  Dev107: Sandy Clay  β=24.3°  H=28cm  sensor=22cm")
    print(f"  Dev108: Sandy Clay Loam  β=19.86°  H=42cm  sensor=30cm")
    print(f"  VG: Carsel & Parrish (1988) pedotransfer")
    print(f"  FoS: Infinite slope Mohr-Coulomb — Rahardjo et al. (2007)")
    print(f"{bar}\n")

    if not os.path.exists(CSV_FILE):
        raise FileNotFoundError(f"CSV not found: {CSV_FILE}")

    data = SlopeDataLoader(CSV_FILE).load()

    print(f"\n{bar}\n  STAGE 1 — Physics Discovery\n{bar}")
    best_model, best_hist, all_results = train_stage1(
        data,
        n_seeds      = 3,
        total_epochs = 18000,
        lr           = 2e-4,
        lam_data     = 800.0,
        lam_richards = 1.0,
        lam_bc_top   = 0.5,
        lam_bc_bot   = 0.1,
        lam_ic       = 0.5,
        lam_smooth   = 0.02,
        lam_prior    = 0.05,
        es_patience  = 5000,
    )

    # Save Stage 1
    pth_s1 = "pinn_slope_singlelayer_stage1.pth"
    torch.save({
        "state_dict":   best_model.state_dict(),
        "architecture": {"hidden":N_HIDDEN,"width":N_WIDTH,"dropout":DROPOUT},
        "site_params":  SITE,
        "stage":        "stage1",
        "learned_vg":   best_model.get_learned_params(),
    }, pth_s1)
    print(f"[Saved] {pth_s1}")

    print(f"\n{bar}\n  STAGE 2 — Rolling Window Adaptation\n{bar}")
    rolling_results = train_stage2(
        best_model, data,
        window_h       = 302.0,
        warmup_epochs  = 500,
        finetune_epochs= 4000,
        lr             = 2e-4,
        lam_data       = 3000.0,
        lam_smooth     = 0.01,
        lam_prior      = 0.5,
        fos_warn_thresh= 1.5,
        es_patience    = 1000,
    )

    # Save Stage 2
    pth_s2 = "pinn_slope_singlelayer_stage2.pth"
    torch.save({
        "state_dict":      best_model.state_dict(),
        "architecture":    {"hidden":N_HIDDEN,"width":N_WIDTH,"dropout":DROPOUT},
        "site_params":     SITE,
        "stage":           "stage2_final",
        "rolling_results": rolling_results,
        "learned_vg":      best_model.get_learned_params(),
    }, pth_s2)
    print(f"[Saved] {pth_s2}")

    print(f"\n{bar}\n  GENERATING FIGURES\n{bar}")
    generate_all_figures(best_model, data, rolling_results, best_hist)

    print(f"\n{bar}\n  FINAL SUMMARY\n{bar}")
    m = compute_metrics_combined(best_model, data)
    print(f"  Test R²   = {m.get('r2_te',  float('nan')):.4f}  (Dev108+Dev107 combined)")
    print(f"  Test RMSE = {m.get('rmse_te',float('nan')):.5f} m³/m³")
    p = best_model.get_learned_params()
    print(f"\n  Learned VG parameters (IoT-fine-tuned):")
    print(f"  Dev107 Sandy Clay:      α={p['alpha_107']:.3f}m⁻¹  n={p['n_vg_107']:.4f}"
          f"  θ_r={p['theta_r_107']:.4f}  θ_s={p['theta_s_107']:.4f}")
    print(f"  Dev108 Sandy Clay Loam: α={p['alpha_108']:.3f}m⁻¹  n={p['n_vg_108']:.4f}"
          f"  θ_r={p['theta_r_108']:.4f}  θ_s={p['theta_s_108']:.4f}")
    if rolling_results:
        worst = min(rolling_results, key=lambda r: r["fos_min"])
        n_warn = sum(r["warning"] for r in rolling_results)
        n_fail = sum(r["failure"] for r in rolling_results)
        print(f"\n  FoS_min = {worst['fos_min']:.3f}  at t=[{worst['t_start_h']:.0f},{worst['t_end_h']:.0f}]h")
        print(f"  Warnings: {n_warn}   Failures: {n_fail}")
    print(bar)

[Device] cpu

  2D PINN — Single-Layer Slope Hydrology
  All parameters from measured texture + geometry
  Dev107: Sandy Clay  β=24.3°  H=28cm  sensor=22cm
  Dev108: Sandy Clay Loam  β=19.86°  H=42cm  sensor=30cm
  VG: Carsel & Parrish (1988) pedotransfer
  FoS: Infinite slope Mohr-Coulomb — Rahardjo et al. (2007)


[Data] Dev107 (Sandy Clay, x=25m, z=22cm): 2719 pts  θ=[0.080,0.463]
[Data] Dev108 (SCL, x=7m, z=30cm): 4131 pts  θ=[0.080,0.539]
[Data] T_MAX_SYNC=1596h  T_MAX_FULL=2118h

  STAGE 1 — Physics Discovery

[PINNSlope] Single-layer slope hydrology
  Dev107 Sandy Clay:     α=2.70m⁻¹  n=1.23  θ_s=0.38
  Dev108 Sandy Clay Loam: α=5.90m⁻¹  n=1.48  θ_s=0.39
  Infinite slope FoS (c'=0): H=[0.28,0.42]m  β=[24.3,19.86]°  φ'=[30.0,30.0]°

─────────────────────────────────────────────────────────────────
[Stage1] Seed=0  lr=0.0002  lam_data=800.0  lam_pde=1.0  epochs=18000
─────────────────────────────────────────────────────────────────
  ep=    1/18000 [warmup]  L= 87.4357  [data=87.4